**LUMINAR**

**Built and running:** auto-trim → scale sweep + simulated viewpoints → SuperPoint + LightGlue + RootSIFT candidates →
Lunar Structural Graph → uniform selection → MAGSAC++ / spread-RANSAC / optional manual seed (coverage-aware, accuracy tiers) →
Huber refinement → non-rigid TPS (only if held-out error improves) → honest metrics → **dashboard** → exports.

**Reserved slots (placeholders, already wired into the pipeline — see Cell 7b):**
`METADATA_ALIGNMENT` · `DEM_RENDER_RELIGHT` (**remove relief and sun**) · `PROPOSE_ALIGNMENTS` · `CHANCE_VERIFICATION` · `DENSE_TIE_POINTS`.
Each is a no-op today; assign `HOOKS['NAME'] = your_function` to install a stage. The dashboard shows which are installed.

**Outputs:** registered.png, checkerboard.png, match_points.csv, report.json, dashboard.png, ZIP.
**Verdicts:** HIGH / MEDIUM / LOW / FAILED, or MANUAL (hand-picked points, not independently verified).

Run order (top to bottom):
**0** install -> **0b** imports + config -> **I0** input library (definitions) ->
**I1** SOURCE (upload picture / upload zip / Google Drive / path) -> **I2** REFERENCE (same four options) ->
**I3** open both + trim lines (sliders) -> **I4** apply trim, orient, scale, save -> **3-8b** pipeline definitions -> **8c** run.
**9** = synthetic self-test (needs only 0, 0b, 3-8b).

Zips with several strips list every strip in a dropdown with its size, pixel size, sun elevation and geometry type; nothing is picked for you.
The geometry used (polar north / polar south / cylindrical / approximate / none for plain pictures) is decided from the label, not assumed.
Hard pair? re-run I3 with other trim lines, or `src_crop=` / `ref_crop=` / `manual_pairs=` in 8c.


In [ ]:
# ===== CELL 0: INSTALL (run once per session) =====
# LightGlue (includes SuperPoint). torch + torchvision already ship with Colab; set Runtime > Change runtime type > T4 GPU for speed.
!pip install -q git+https://github.com/cvg/LightGlue.git
!pip install -q scikit-image scipy matplotlib opencv-python-headless
print("Install done. If Colab asks to restart the runtime, restart it and run from Cell 0b.")


In [ ]:
# ===== CELL 0b: IMPORTS + CONFIG (run after Cell 0, before everything else) =====
import os, glob, json, csv, time, shutil, zipfile, hashlib, textwrap
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np, cv2
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.spatial import cKDTree, ConvexHull
from scipy.optimize import least_squares
from scipy.interpolate import RBFInterpolator
from skimage.metrics import structural_similarity as ssim

try:
    import torch
    from lightglue import LightGlue, SuperPoint
    from lightglue.utils import rbd
    print("SuperPoint + LightGlue OK | device:", "cuda" if torch.cuda.is_available() else "cpu (slow - use a T4 GPU runtime)")
except Exception as e:
    print("WARNING: SuperPoint/LightGlue not available (", type(e).__name__, e, ") - pipeline will use RootSIFT only. Run Cell 0.")

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

CFG = {
    'auto_trim': True, 'work_max_side': 1500, 'viewpoint_sim': 'auto',
    'scales': [0.25, 0.5, 1.0, 2.0], 'vp_scales': [0.5, 1.0], 'sweep_tilts': [1.0, 1.5],
    'max_work_pixels': 8000000, 'min_scale_inliers': 15,
    'sift_nfeatures': 8000, 'sift_ratio': 0.75, 'sift_sweep_px': 10.0, 'clahe': True,
    'sp_max_kp': 2048, 'lg_filter_threshold': 0.1, 'vp_both_directions': False,
    'tilts_deep': [1.0, 2.0], 'tilts_sift': [1.0, 1.5, 2.2], 'rot_base': 72.0, 'max_views': 12,
    'use_sift_pool': True, 'dedupe_px': 3.0, 'max_candidates': 4000, 'coarse_px': 25.0,
    'lsg_iters': 5, 'lsg_k': 12, 'lsg_tol': 0.15, 'lsg_min_area_ratio': 0.05, 'lsg_min_score': 0.35, 'lsg_min_keep': 12,
    'grid': 4, 'max_per_cell': 80, 'px_tiers': [2.0, 4.0, 8.0, 16.0], 'min_tier_inliers': 12,
    'min_cov': 0.30, 'cov_grid': 4, 'spread_iters': 2000, 'huber_fscale': 1.5, 'min_refine_inliers': 10,
    'nr_enable': True, 'nr_min_pts': 15, 'nr_smoothing': 0.2, 'nr_gain': 0.85, 'nr_fade': 0.25, 'cv_folds': 5,
    'out_dir': 'luminar_output', 'show': True,
}


In [ ]:
# ===== CELL I0: INPUT + GEO LIBRARY (definitions only - run once, nothing happens yet) =====
# Handles: pictures (png/jpg/tif/webp), PDS4 .img + .xml, zips holding either, Google Drive, a path.
# Detects what kind of image it is (PDS4 label -> polar north / polar south / cylindrical / unknown, or a plain picture)
# and only then applies the geometry that fits that type.
import re
from IPython.display import display, clear_output

IMG_EXT    = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp', '.webp')
RASTER_EXT = ('.img',) + IMG_EXT
DTYPES     = {"UnsignedLSB2": "<u2", "UnsignedMSB2": ">u2", "SignedLSB2": "<i2", "SignedMSB2": ">i2",
              "UnsignedByte": "u1", "SignedByte": "i1", "IEEE754LSBSingle": "<f4", "IEEE754MSBSingle": ">f4",
              "UnsignedLSB4": "<u4", "SignedLSB4": "<i4"}
R_MOON     = 1737.4          # km
WORK_ROOT  = "/content/luminar_work"
os.makedirs(WORK_ROOT, exist_ok=True)

def _xt(e): return e.tag.split('}')[-1]

def _num(x):
    try: return float(x)
    except Exception: return None

# ------------------------------------------------------------------ PDS4 label -> dict
def parse_pds4_label(src):
    """src = path or bytes of a PDS4 .xml label. Returns a dict, or None if it is not a 2-D image label."""
    try:
        raw = bytes(src) if isinstance(src, (bytes, bytearray)) else open(src, 'rb').read()
        k = min([i for i in (raw.find(b'<?xml'), raw.find(b'<Product_')) if i >= 0] or [0])
        root = ET.fromstring(raw[k:])          # tolerates junk before the XML (e.g. text copied from a browser view)
    except Exception:
        return None
    arr = next((e for e in root.iter() if _xt(e) in ('Array_2D_Image', 'Array_2D')), None)
    if arr is None:
        return None
    axes = []
    for ax in arr.iter():
        if _xt(ax) == 'Axis_Array':
            k = {_xt(c): (c.text or '').strip() for c in ax}
            axes.append((int(k.get('sequence_number', len(axes) + 1)), k.get('axis_name', ''), int(k['elements'])))
    axes.sort()
    if len(axes) != 2:
        return None
    first = {}
    for e in root.iter():
        if e.text and e.text.strip() and _xt(e) not in first:
            first[_xt(e)] = e.text.strip()
    kids = {_xt(c): (c.text or '').strip() for c in arr}
    dt = next((c.text.strip() for c in arr.iter() if _xt(c) == 'data_type'), 'UnsignedLSB2')
    corners = None
    for blk in ('Refined_Corner_Coordinates', 'System_Level_Coordinates', None):
        scope = root if blk is None else next((e for e in root.iter() if _xt(e) == blk), None)
        if scope is None:
            continue
        v = {_xt(c): _num(c.text) for c in scope.iter() if c.text}
        try:
            corners = {k: (v[f'{n}_latitude'], v[f'{n}_longitude'])
                       for k, n in (('UL', 'upper_left'), ('UR', 'upper_right'), ('LL', 'lower_left'), ('LR', 'lower_right'))}
            if any(x is None for c in corners.values() for x in c):
                corners = None
                continue
            break
        except KeyError:
            corners = None
    return dict(rows=axes[0][2], cols=axes[1][2], axes=(axes[0][1], axes[1][1]), dtype=DTYPES.get(dt, '<u2'),
                offset=int(float(kids.get('offset', 0) or 0)),
                col_major=('first index fastest' in first.get('axis_index_order', '').lower()),
                name=first.get('file_name', ''), md5=first.get('md5_checksum'),
                res_m=_num(first.get('pixel_resolution')), sun_elev=_num(first.get('sun_elevation')),
                projection=first.get('projection', ''), area=first.get('area', ''), corners=corners,
                instrument=first.get('title', ''))

def classify_projection(meta):
    """'polar_north' | 'polar_south' | 'cylindrical' | 'unknown' | 'none' (no geo information)."""
    c = (meta or {}).get('corners')
    if not c:
        return 'none'
    p, a = (meta.get('projection') or '').lower(), (meta.get('area') or '').lower()
    lats = [v[0] for v in c.values()]
    if 'polar' in p or 'stereo' in p:
        return 'polar_south' if ('south' in a or ('north' not in a and np.mean(lats) < 0)) else 'polar_north'
    if any(k in p for k in ('cylind', 'mercator', 'equirect', 'plate', 'simple')):
        return 'cylindrical'
    if max(abs(x) for x in lats) > 85 and (max(lats) - min(lats)) > 15:       # label silent, geometry says "runs to the pole"
        return 'polar_south' if np.mean(lats) < 0 else 'polar_north'
    return 'unknown'

# ------------------------------------------------------------------ raster access (never loads a whole big strip)
class Raster:
    def __init__(self, path, meta=None):
        self.path, self.meta = path, meta
        if meta:                                              # PDS4 raw binary + label
            self.kind = 'pds4'
            rows, cols = meta['rows'], meta['cols']
            need = meta['offset'] + rows * cols * np.dtype(meta['dtype']).itemsize
            have = os.path.getsize(path)
            if have != need:
                raise ValueError(f"{os.path.basename(path)}: file is {have} bytes but its label says {need}. "
                                 "The download/unzip is incomplete or the label belongs to another file.")
            shape = (cols, rows) if meta['col_major'] else (rows, cols)
            mm = np.memmap(path, dtype=meta['dtype'], mode='r', offset=meta['offset'], shape=shape)
            self._a = mm.T if meta['col_major'] else mm
            self.display_ready = False
        else:                                                 # ordinary picture
            self.kind = 'image'
            im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
            if im is None:
                raise ValueError(f"OpenCV cannot read '{path}'. Re-save it as PNG/JPG/TIFF and try again.")
            if im.ndim == 3:
                im = cv2.cvtColor(im[..., :3], cv2.COLOR_BGR2GRAY)
            self._a = im
            self.display_ready = (im.dtype == np.uint8)
        self.shape = self._a.shape[:2]

    @property
    def res_m(self): return (self.meta or {}).get('res_m')

    def read(self, r0, r1, c0, c1, sr=1, sc=1):
        return np.asarray(self._a[r0:r1:sr, c0:c1:sc], dtype=np.float32)

    def valid(self, a):
        if self.kind == 'image':
            return np.ones(a.shape, bool)
        return np.isfinite(a) & (a > 0)

    def to_u8(self, a):
        """Contrast-stretch (1-99 %) over the valid pixels; 8-bit pictures are left untouched."""
        if self.display_ready:
            return np.clip(a, 0, 255).astype(np.uint8)
        v = self.valid(a)
        if not v.any():
            return np.zeros(a.shape, np.uint8)
        lo, hi = np.percentile(a[v], (1, 99))
        if hi - lo < 1e-9:
            hi = lo + 1
        return (np.clip((a - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

# ------------------------------------------------------------------ geometry, chosen by image type
class Geo:
    """pixel (row, col) of the FULL raster -> (lat, lon). Method depends on the detected projection:
       polar_north / polar_south : polar stereographic (interpolated in the projected plane)
       cylindrical / unknown     : bilinear in lat/lon between the four corners (approximate; 'unknown' is flagged)"""
    def __init__(self, meta, shape, method='A'):
        self.kind = classify_projection(meta)
        self.c, self.rows, self.cols = meta['corners'], shape[0], shape[1]
        self.res_km = (meta.get('res_m') or 0) / 1000.0
        self.method = method if (method == 'B' and self.res_km) else 'A'
        self.hemi = -1 if self.kind == 'polar_south' else 1
        if self.kind.startswith('polar'):
            self.xy = {k: self._to_xy(*v) for k, v in self.c.items()}
            d = self.xy['LL'] - self.xy['UL']
            self.row_u = d / np.linalg.norm(d)
        else:
            ul = self.c['UL'][1]
            self.lonu = {k: ul + ((v[1] - ul + 180) % 360 - 180) for k, v in self.c.items()}

    def _to_xy(self, lat, lon):
        th, lam = np.radians(90 - self.hemi * lat), np.radians(lon)
        r = 2 * R_MOON * np.tan(th / 2)
        return np.array([r * np.sin(lam), -self.hemi * r * np.cos(lam)])

    def _from_xy(self, xy):
        r = np.hypot(*xy)
        lat = self.hemi * (90 - np.degrees(2 * np.arctan(r / (2 * R_MOON))))
        return lat, np.degrees(np.arctan2(xy[0], -self.hemi * xy[1]))

    def pix2ll(self, row, col, method=None):
        fr, fc = row / self.rows, col / self.cols
        if self.kind.startswith('polar'):
            if (method or self.method) == 'B':
                p = self.xy['UL'] + fc * (self.xy['UR'] - self.xy['UL']) + row * self.res_km * self.row_u
            else:
                top = self.xy['UL'] + fc * (self.xy['UR'] - self.xy['UL'])
                bot = self.xy['LL'] + fc * (self.xy['LR'] - self.xy['LL'])
                p = top + fr * (bot - top)
            lat, lon = self._from_xy(p)
        else:
            L = lambda k, i: (self.c[k][0] if i == 0 else self.lonu[k])
            top = lambda i: L('UL', i) + fc * (L('UR', i) - L('UL', i))
            bot = lambda i: L('LL', i) + fc * (L('LR', i) - L('LL', i))
            lat, lon = top(0) + fr * (bot(0) - top(0)), top(1) + fr * (bot(1) - top(1))
        return float(lat), float((lon + 180) % 360 - 180)

    def orientation(self, row, col):
        """Local Jacobian from finite differences -> north angle (deg clockwise from image top), mirrored?, km/px."""
        d = max(20.0, 0.01 * max(self.rows, self.cols))
        la0, lo0 = self.pix2ll(row, col)
        def rel(r, c):
            la, lo = self.pix2ll(r, c)
            return np.array([np.radians((lo - lo0 + 180) % 360 - 180) * R_MOON * np.cos(np.radians(la0)),
                             np.radians(la - la0) * R_MOON])
        dc = (rel(row, col + d) - rel(row, col - d)) / (2 * d)       # (east, north) km per column step
        dr = (rel(row + d, col) - rel(row - d, col)) / (2 * d)       # (east, north) km per row step
        J = np.column_stack([dc, dr])
        norms = np.linalg.norm(J, axis=0)
        Jn = J / np.where(norms > 0, norms, 1)
        v = np.linalg.solve(Jn, [0.0, 1.0])                          # image direction of north
        return dict(north_deg=float(np.degrees(np.arctan2(v[0], -v[1])) % 360), mirrored=bool(np.linalg.det(Jn) > 0),
                    Jn=Jn, km_per_px=norms, lat=la0, lon=lo0)

def make_geo(raster, method='A'):
    m = raster.meta
    return Geo(m, raster.shape, method) if (m and m.get('corners')) else None

def make_north_up(im, Jn):
    """Rotate/mirror a picture so north is up and east is right (canvas grows to hold the rotated image)."""
    h, w = im.shape[:2]
    Aff = np.linalg.inv(Jn) @ np.diag([1.0, -1.0])                  # output (east right, south down) -> source offsets
    c = np.array([(w - 1) / 2.0, (h - 1) / 2.0])                    # pixel-centre convention
    corners = np.array([[0, 0], [w - 1, 0], [0, h - 1], [w - 1, h - 1]], float) - c
    ext = np.abs((np.linalg.inv(Aff) @ corners.T).T).max(0)
    W_, H_ = int(round(2 * ext[0])) + 1, int(round(2 * ext[1])) + 1
    t = c - Aff @ np.array([(W_ - 1) / 2.0, (H_ - 1) / 2.0])
    M = np.hstack([Aff, t[:, None]]).astype(np.float32)
    return cv2.warpAffine(im, M, (W_, H_), flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP, borderValue=0)

# ------------------------------------------------------------------ finding candidate images (files and zip members)
def _fmt_option(name, size, meta, tag=''):
    if meta:
        kind = classify_projection(meta)
        bits = [name, f"{meta['rows']}x{meta['cols']} px"]
        if meta.get('res_m'): bits.append(f"{meta['res_m']:g} m/px")
        if meta.get('sun_elev') is not None: bits.append(f"sun {meta['sun_elev']:.0f} deg")
        bits.append(kind)
    else:
        bits = [name]
    bits.append(f"{size / 1e6:.1f} MB")
    return ' | '.join(bits) + (f'  {tag}' if tag else '')

def _tag_for(name):
    return '[preview chip]' if re.search(r'(^|/)browse(/|$)', name.replace('\\', '/').lower()) else ''

def list_options(paths):
    """Every image / PDS4 strip inside the given files (zips are looked into, not extracted)."""
    opts = []
    for p in paths:
        ext = os.path.splitext(p)[1].lower()
        if ext == '.zip':
            with zipfile.ZipFile(p) as z:
                names = z.namelist(); low = {n.lower(): n for n in names}
                for n in names:
                    e = os.path.splitext(n)[1].lower()
                    if e not in RASTER_EXT:
                        continue
                    meta = None
                    if e == '.img':
                        xn = low.get(n[:-4].lower() + '.xml')
                        meta = parse_pds4_label(z.read(xn)) if xn else None
                    opts.append(dict(kind='zip', zip=p, member=n, meta=meta, size=z.getinfo(n).file_size,
                                     label=_fmt_option(os.path.basename(n) + ('' if e != '.img' or meta else '  (no label found)'),
                                                       z.getinfo(n).file_size, meta, _tag_for(n)) + f"   <{os.path.basename(p)}>"))
        elif ext in RASTER_EXT:
            meta = None
            if ext == '.img':
                xml = next((p[:-4] + x for x in ('.xml', '.XML') if os.path.exists(p[:-4] + x)), None)
                meta = parse_pds4_label(xml) if xml else None
            opts.append(dict(kind='file', path=p, meta=meta, size=os.path.getsize(p),
                             label=_fmt_option(os.path.basename(p) + ('' if ext != '.img' or meta else '  (no label found)'),
                                               os.path.getsize(p), meta)))
    opts.sort(key=lambda o: (o['label'].find('[preview chip]') >= 0, 0 if o['meta'] else 1, o['label']))
    return opts

class Picker:
    """Shows a dropdown of every candidate; nothing is chosen for you when there is more than one."""
    def __init__(self, role, options):
        if not options:
            raise FileNotFoundError(f"No image or PDS4 .img found for the {role}. Check what you uploaded / the folder.")
        self.role, self.options, self.dd, self.index = role, options, None, (0 if len(options) == 1 else None)
        try:
            import ipywidgets as W
            self.dd = W.Dropdown(options=[(o['label'], i) for i, o in enumerate(options)], index=self.index,
                                 description=role.upper() + ':', layout=W.Layout(width='98%'),
                                 style={'description_width': '90px'})
        except Exception:
            pass
        if len(options) == 1:
            print(f"{role}: only one candidate -> {options[0]['label']}")
        else:
            print(f"{role}: {len(options)} candidates found. Choose one in the dropdown below, THEN run the next cell.")
            if self.dd is not None:
                display(self.dd)
            else:
                for i, o in enumerate(options): print(f"  [{i}] {o['label']}")
                print("ipywidgets unavailable: set  <picker>.index = N  in a cell, then continue.")

    @property
    def choice(self):
        i = self.dd.value if self.dd is not None else self.index
        if i is None:
            raise ValueError(f"Choose the {self.role} in the dropdown first (the cell above), then re-run this cell.")
        return self.options[i]

def acquire(role, mode, drive_folder='', path=''):
    """mode: 'Upload (auto-detect)' | 'Google Drive file' | 'File path in Colab' -> Picker. Pictures, zips and .img+.xml are told apart by content, not by the mode."""
    d = os.path.join(WORK_ROOT, role + '_in'); os.makedirs(d, exist_ok=True)
    paths = []
    if mode.startswith('Upload'):
        from google.colab import files
        print(f"Select the {role.upper()} file(s): a picture (png/jpg/tif/webp), a zip, or a PDS4 .img together with its .xml - the type is detected automatically ...")
        up = files.upload()
        if not up:
            raise ValueError("Nothing was uploaded. Re-run this cell and choose a file.")
        for n in up:
            dest = os.path.join(d, os.path.basename(n)); shutil.move(n, dest); paths.append(dest)
    elif mode.startswith('Google Drive'):
        if not os.path.exists('/content/drive/MyDrive'):
            try:
                from google.colab import drive
                drive.mount('/content/drive')
            except Exception as e:
                raise RuntimeError(f"Could not mount Google Drive ({e}). Use 'Upload zip' / 'File path in Colab' instead "
                                   "(drag the file into the folder panel on the left).")
        base = os.path.join('/content/drive/MyDrive', drive_folder.strip('/'))
        found = []
        for depth in ('*', '*/*'):
            found += glob.glob(os.path.join(base, depth))
        paths = sorted([f for f in found if os.path.isfile(f) and os.path.splitext(f)[1].lower() in ('.zip',) + RASTER_EXT],
                       key=os.path.getmtime, reverse=True)
        print(f"{len(paths)} zip/image file(s) found in {base} (newest first)")
    elif mode.startswith('File path'):
        if not path:
            raise ValueError("Type the file or folder path in the 'PATH' field of the form above.")
        paths = sorted(glob.glob(os.path.join(path, '*'))) if os.path.isdir(path) else [path]
        if not paths or not os.path.exists(paths[0]):
            raise FileNotFoundError(f"Nothing at {path}")
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return Picker(role, list_options(paths))

def open_raster(opt):
    """Extract only the chosen strip (and its label) if it is inside a zip, then open it."""
    if opt['kind'] == 'zip':
        out = os.path.join(WORK_ROOT, 'extracted'); os.makedirs(out, exist_ok=True)
        with zipfile.ZipFile(opt['zip']) as z:
            names = {n.lower(): n for n in z.namelist()}
            z.extract(opt['member'], out)
            xn = names.get(opt['member'][:-4].lower() + '.xml') if opt['member'].lower().endswith('.img') else None
            if xn: z.extract(xn, out)
        path = os.path.join(out, opt['member'])
    else:
        path = opt['path']
    meta = opt['meta']
    if path.lower().endswith('.img') and meta is None:
        raise ValueError("This .img has no PDS4 .xml label next to it, so its size/format are unknown. Upload the .xml too.")
    return Raster(path, meta if path.lower().endswith('.img') else None)

# ------------------------------------------------------------------ trimming: suggestion + interactive lines
def suggest_box(raster, geo=None, side=4000, lat_limit=85.0, min_valid=0.98, small_px=6e6, step=8):
    """Most textured block (craters/ridges) of a big raster; the whole picture if it is small.
    Skips blocks with no-data edges and (when geo is known) blocks closer to a pole than lat_limit."""
    h, w = raster.shape
    if h * w <= small_px:
        return (0, 0, w, h)
    sh, sw = min(h, side), min(w, side)
    def scan(use_lat, use_valid):
        best = None
        for y in range(0, h - sh + 1, max(1, sh // 2)):
            for x in range(0, w - sw + 1, max(1, sw // 2)):
                if use_lat and geo is not None and abs(geo.pix2ll(y + sh / 2, x + sw / 2)[0]) > lat_limit:
                    continue
                s = raster.read(y, y + sh, x, x + sw, step, step)
                v = raster.valid(s)
                if use_valid and v.mean() < min_valid:
                    continue
                if v.sum() < 16:
                    continue
                lo, hi = np.percentile(s[v], (1, 99))
                if hi - lo < 1e-9:
                    continue
                n = np.clip((s - lo) / (hi - lo), 0, 1).astype(np.float32)
                sc = float(cv2.Laplacian(n, cv2.CV_32F).var())
                if best is None or sc > best[0]:
                    best = (sc, (x, y, x + sw, y + sh))
        return best
    best = scan(True, True) or scan(False, True) or scan(False, False)
    return best[1] if best else ((w - sw) // 2, (h - sh) // 2, (w + sw) // 2, (h + sh) // 2)

class TrimSel:
    def __init__(self, raster, box=None):
        h, w = raster.shape
        self.raster, self.box = raster, tuple(int(v) for v in (box or (0, 0, w, h)))   # (x0, y0, x1, y1)

def parse_box(txt, raster):
    v = [int(float(t)) for t in re.split(r'[,\s]+', txt.strip()) if t]
    if len(v) != 4:
        raise ValueError("Manual box must be 4 numbers: x0,y0,x1,y1 (pixels of the ORIGINAL image)")
    h, w = raster.shape
    x0, y0, x1, y1 = max(0, v[0]), max(0, v[1]), min(w, v[2]), min(h, v[3])
    if x1 - x0 < 32 or y1 - y0 < 32:
        raise ValueError("Manual box is too small / outside the image")
    return (x0, y0, x1, y1)

def trim_ui(raster, role, geo=None, start_box=None, lat_limit=85.0):
    """Two red lines (top / bottom) and two green lines (left / right) over an overview of the whole image.
    Drag the sliders or type exact numbers; the right panel shows exactly what will be kept."""
    sel = TrimSel(raster, start_box)
    try:
        import ipywidgets as W
    except Exception:
        print("ipywidgets not available -> using the starting box. Set a manual box in the form instead.")
        return sel
    h, w = raster.shape
    sr, sc = max(1, int(np.ceil(h / 1000))), max(1, int(np.ceil(w / 700)))
    ov = raster.to_u8(raster.read(0, h, 0, w, sr, sc))
    x0, y0, x1, y1 = sel.box
    lay = W.Layout(width='98%')
    rows = W.IntRangeSlider(value=(y0, y1), min=0, max=h, step=1, description='Rows', continuous_update=False, layout=lay)
    cols = W.IntRangeSlider(value=(x0, x1), min=0, max=w, step=1, description='Cols', continuous_update=False, layout=lay)
    tb = {k: W.IntText(value=v, description=k, layout=W.Layout(width='150px')) for k, v in
          (('top', y0), ('bottom', y1), ('left', x0), ('right', x1))}
    b_sug, b_full = W.Button(description='Suggest best block'), W.Button(description='Whole image')
    info, out, busy = W.HTML(), W.Output(), {'v': False}

    def current():
        (a, b), (c, d) = rows.value, cols.value
        return (int(c), int(a), int(d), int(b))

    def redraw():
        bx = current(); sel.box = bx; X0, Y0, X1, Y1 = bx
        txt = f"<b>{role.upper()}</b> keep rows {Y0}-{Y1}, cols {X0}-{X1} &rarr; {X1 - X0} x {Y1 - Y0} px"
        if raster.res_m:
            txt += f" &asymp; {(X1 - X0) * raster.res_m / 1000:.1f} x {(Y1 - Y0) * raster.res_m / 1000:.1f} km"
        if geo is not None:
            la, lo = geo.pix2ll((Y0 + Y1) / 2, (X0 + X1) / 2)
            txt += f" &middot; centre lat {la:.3f}, lon {lo:.3f} <i>({geo.kind})</i>"
        if (X1 - X0) * (Y1 - Y0) > 36e6:
            txt += " &middot; <span style='color:#b45309'>big window: it will be read at reduced resolution</span>"
        if (X1 - X0) < 64 or (Y1 - Y0) < 64:
            txt += " &middot; <span style='color:#b91c1c'>too small to match</span>"
        info.value = txt
        with out:
            clear_output(wait=True)
            fig = plt.figure(figsize=(11, 5)); gs = fig.add_gridspec(1, 2, width_ratios=[1, 3])
            a0 = fig.add_subplot(gs[0]); a0.imshow(ov, cmap='gray', aspect='auto', extent=(0, w, h, 0))
            a0.axhline(Y0, color='red', lw=1.5); a0.axhline(Y1, color='red', lw=1.5)
            if X0 > 0: a0.axvline(X0, color='lime', lw=1.2)
            if X1 < w: a0.axvline(X1, color='lime', lw=1.2)
            a0.set_title('overview (red = top/bottom)', fontsize=9)
            a1 = fig.add_subplot(gs[1])
            ps, pc = max(1, int(np.ceil((Y1 - Y0) / 800))), max(1, int(np.ceil((X1 - X0) / 800)))
            win = raster.to_u8(raster.read(Y0, Y1, X0, X1, ps, pc))
            a1.imshow(win, cmap='gray', extent=(X0, X1, Y1, Y0)); a1.set_title('what will be kept', fontsize=9)
            plt.tight_layout(); plt.show()

    def set_all(bx):
        busy['v'] = True
        rows.value, cols.value = (bx[1], bx[3]), (bx[0], bx[2])
        tb['top'].value, tb['bottom'].value, tb['left'].value, tb['right'].value = bx[1], bx[3], bx[0], bx[2]
        busy['v'] = False
        redraw()

    def on_slider(_):
        if busy['v']: return
        bx = current(); busy['v'] = True
        tb['top'].value, tb['bottom'].value, tb['left'].value, tb['right'].value = bx[1], bx[3], bx[0], bx[2]
        busy['v'] = False; redraw()

    def on_text(_):
        if busy['v']: return
        t, b = sorted((max(0, min(h, tb['top'].value)), max(0, min(h, tb['bottom'].value))))
        l, r = sorted((max(0, min(w, tb['left'].value)), max(0, min(w, tb['right'].value))))
        if b - t < 2 or r - l < 2: return
        set_all((l, t, r, b))

    rows.observe(on_slider, names='value'); cols.observe(on_slider, names='value')
    for t in tb.values(): t.observe(on_text, names='value')
    b_sug.on_click(lambda _b: set_all(suggest_box(raster, geo, lat_limit=lat_limit)))
    b_full.on_click(lambda _b: set_all((0, 0, w, h)))
    display(W.VBox([info, rows, cols, W.HBox(list(tb.values())), W.HBox([b_sug, b_full]), out]))
    redraw()
    sel.ui = dict(rows=rows, cols=cols, set_all=set_all)
    return sel

def trim_setup(mode, raster, geo, role, manual='', lat_limit=85.0):
    if mode.startswith('Manual'):
        return TrimSel(raster, parse_box(manual, raster))
    if mode.startswith('Full'):
        return TrimSel(raster)
    start = suggest_box(raster, geo, lat_limit=lat_limit) if (role == 'source' or mode.startswith('Interactive')) else None
    if mode.startswith('Auto'):
        return TrimSel(raster, start)
    return trim_ui(raster, role, geo, start, lat_limit)

# ------------------------------------------------------------------ cut, scale, orient, save
def prepare_image(raster, box, geo, role, target_res_m=None, max_side=2000, north_up=True, max_read_px=36e6):
    x0, y0, x1, y1 = box
    n = (x1 - x0) * (y1 - y0)
    s0 = 1 if n <= max_read_px else int(np.ceil(np.sqrt(n / max_read_px)))
    u8 = raster.to_u8(raster.read(y0, y1, x0, x1, s0, s0))
    res_full = raster.res_m
    ori = None
    if geo is not None:
        ori = geo.orientation((y0 + y1) / 2, (x0 + x1) / 2)
        if not res_full:
            res_full = float(np.mean(ori['km_per_px']) * 1000)
    res_now = res_full * s0 if res_full else None
    f = 1.0
    if target_res_m and res_now and target_res_m > res_now:
        f = res_now / target_res_m
    f = min(f, max_side / max(u8.shape))
    if f < 0.999:
        u8 = cv2.resize(u8, None, fx=f, fy=f, interpolation=cv2.INTER_AREA)
    applied_north_up = False
    if ori is not None and north_up and (ori['mirrored'] or min(ori['north_deg'], 360 - ori['north_deg']) > 1.0):
        u8 = make_north_up(u8, ori['Jn']); applied_north_up = True
    path = os.path.join(WORK_ROOT, f"{role}_crop.png")
    cv2.imwrite(path, u8)
    res_out = (res_now / f) if res_now else None
    return dict(path=path, shape=u8.shape[:2], box=box, scale=f, res_m_in=raster.res_m, res_m_out=res_out,
                orientation=ori, north_up=applied_north_up, geo_kind=(geo.kind if geo else None), read_step=s0)

def describe_location(geo, box):
    """Printable coordinates of the kept window + tips that depend on the image type."""
    x0, y0, x1, y1 = box
    print("   corners of the kept window (lat, lon):")
    for nm, (r, c) in (("top-left", (y0, x0)), ("top-right", (y0, x1)), ("bottom-left", (y1, x0)), ("bottom-right", (y1, x1))):
        la, lo = geo.pix2ll(r, c); print(f"     {nm:13s} {la:9.4f}  {lo:9.4f}")
    la, lo = geo.pix2ll((y0 + y1) / 2, (x0 + x1) / 2)
    print(f"   centre -> paste into QuickMap search: {la:.4f}, {lo:.4f}")
    if geo.kind.startswith('polar') and geo.res_km:
        la2, lo2 = geo.pix2ll((y0 + y1) / 2, (x0 + x1) / 2, method='B')
        gap = np.linalg.norm(geo._to_xy(la, lo) - geo._to_xy(la2, lo2))
        print(f"   second estimate (rows x pixel size): {la2:.4f}, {lo2:.4f}  ({gap:.1f} km apart; if the craters are not at the first spot, look near this one)")
    tip = {'polar_north': "QuickMap projection: 'Stereographic (North Pole)'",
           'polar_south': "QuickMap projection: 'Stereographic (South Pole)'",
           'cylindrical': ("QuickMap projection: 'Equidistant Cylindrical'" if abs(la) < 75 else
                           "above 75 deg latitude use a polar stereographic projection in QuickMap"),
           'unknown': "projection not stated in the label: coordinates are approximate (linear between corners)"}[geo.kind]
    print("   " + tip)


In [ ]:
# ===== CELL: CONVERT a downloaded .IMG (PDS3 or PDS4) to a PNG you can open / upload =====
IMG_INPUT = ""     #@param {type:"string"}
LABEL_FILE = ""    #@param {type:"string"}
MAX_SIDE   = 4000  #@param {type:"integer"}
ROWS       = ""    #@param {type:"string"}
# IMG_INPUT : path of the .IMG in Colab (e.g. /content/M123456789LE.IMG), or a Google Drive share link ("Anyone with the link").
#             Leave blank to get an upload button.
# LABEL_FILE: only if the label is a SEPARATE file (.LBL / .xml) with a different name. Usually blank - it is found automatically.
# MAX_SIDE  : the PNG is shrunk so its longest side is at most this many pixels (a whole strip can be 50 000+ rows).
# ROWS      : optional "start:end" to keep only those rows, e.g. 20000:24000 (blank = everything).
import os, re, glob, shutil
import numpy as np, cv2

def _pds3_label(path, label_path=None):
    """Read a PDS3 label (inside the .IMG, or a detached .LBL). Returns (dict, image_file, byte_offset) or None."""
    src = label_path or path
    with open(src, 'rb') as f: head = f.read(4_000_000)
    txt = head.decode('latin-1', errors='replace')
    if 'PDS_VERSION_ID' not in txt[:2000].upper() and 'RECORD_TYPE' not in txt[:2000].upper():
        return None
    m = re.search(r'^\s*END\s*$', txt, re.M)
    if not m:
        return None
    txt = txt[:m.start()]
    kv = {}
    for line in txt.splitlines():
        mm = re.match(r'^\s*([\^A-Za-z0-9_:]+)\s*=\s*(.+?)\s*$', line)
        if mm: kv.setdefault(mm.group(1).upper(), mm.group(2).strip())
    def num(k, default=None):
        v = kv.get(k)
        try: return int(float(re.match(r'[-+0-9.eE]+', v.strip('"')).group(0)))
        except Exception: return default
    rec = num('RECORD_BYTES', 1)
    ptr = kv.get('^IMAGE', '')
    img_file, off = path, 0
    fn = re.search(r'"([^"]+)"', ptr)
    n = re.search(r'(?<![A-Za-z0-9_.])(\d+)(?![A-Za-z0-9_.])\s*(<BYTES>)?\s*\)?\s*$', ptr) or re.search(r'(\d+)', ptr.split(',')[-1] if ',' in ptr else ptr)
    if fn:
        cand = os.path.join(os.path.dirname(path), fn.group(1))
        if os.path.exists(cand): img_file = cand
        else:
            hit = glob.glob(os.path.join(os.path.dirname(path), '**', fn.group(1)), recursive=True) + \
                  [p for p in glob.glob(os.path.join(os.path.dirname(path), '*')) if os.path.basename(p).lower() == fn.group(1).lower()]
            if hit: img_file = hit[0]
    if n:
        k = int(n.group(1)); off = (k - 1) if n.group(2) else (k - 1) * rec
    elif ptr == '':
        off = (num('LABEL_RECORDS', 0) or 0) * rec
    lines, samples = num('LINES'), num('LINE_SAMPLES')
    if not lines or not samples:
        return None
    return dict(lines=lines, samples=samples, bits=num('SAMPLE_BITS', 8), type=kv.get('SAMPLE_TYPE', 'UNSIGNED_INTEGER').strip('"').upper(),
                bands=num('BANDS', 1) or 1, prefix=num('LINE_PREFIX_BYTES', 0) or 0, suffix=num('LINE_SUFFIX_BYTES', 0) or 0,
                null=num('CORE_NULL'), compressed=any(k in kv for k in ('ENCODING_TYPE', 'IMAGE_COMPRESSION', 'COMPRESSION_TYPE')) and
                'UNCOMPRESSED' not in ' '.join(kv.get(k, '') for k in ('ENCODING_TYPE', 'IMAGE_COMPRESSION', 'COMPRESSION_TYPE')).upper(),
                target=kv.get('TARGET_NAME', ''), product=kv.get('PRODUCT_ID', ''),
                enc=' '.join(kv.get(k, '') for k in ('ENCODING_TYPE', 'IMAGE_COMPRESSION', 'COMPRESSION_TYPE')).strip()), img_file, off

def _pds3_dtype(info):
    t, b = info['type'], info['bits']
    little = t.startswith(('LSB', 'PC_', 'VAX')) and not t.startswith('VAX_R')
    if 'REAL' in t or 'FLOAT' in t:
        return ('<' if little else '>') + {32: 'f4', 64: 'f8'}.get(b, 'f4')
    signed = ('INTEGER' in t) and ('UNSIGNED' not in t) and not t.startswith('PC_UNSIGNED')
    if b == 8: return 'i1' if signed else 'u1'
    return ('<' if little else '>') + {16: 'i2' if signed else 'u2', 32: 'i4' if signed else 'u4'}[b]

def _open_any(path, label_path=None):
    """-> 2-D array-like (memmap), nodata value, description."""
    p3 = _pds3_label(path, label_path)
    if not p3 and not label_path:                                   # no header inside the file -> look for a detached .LBL next to it
        side = next((path[:-4] + e for e in ('.lbl', '.LBL', '.Lbl') if os.path.exists(path[:-4] + e)), None)
        if side: p3 = _pds3_label(path, side)
    if p3:
        info, f, off = p3
        if info['compressed']:
            raise ValueError("This PDS3 file is compressed inside (label says " + str(info['enc']) + "). It needs ISIS/GDAL to open; "
                             "download the uncompressed EDR/CDR version of the product instead.")
        dt = np.dtype(_pds3_dtype(info)); L, S, B = info['lines'], info['samples'], info['bands']
        rec = info['prefix'] + S * dt.itemsize + info['suffix']
        raw = np.memmap(f, dtype='u1', mode='r', offset=off, shape=(L * B, rec))
        arr = raw[:, info['prefix']:info['prefix'] + S * dt.itemsize].view(dt)
        arr = arr[:L] if B > 1 else arr
        return arr, info['null'], f"PDS3 {info['product']} {L} lines x {S} samples, {info['bits']}-bit {info['type']}"
    # PDS4 (.img + .xml) - uses the library from Cell I0 when present
    xml = label_path or next((path[:-4] + e for e in ('.xml', '.XML') if os.path.exists(path[:-4] + e)), None)
    if xml and 'parse_pds4_label' in globals():
        m = parse_pds4_label(xml)
        if m:
            r = Raster(path, m)
            return r._a, None, f"PDS4 {m['rows']} x {m['cols']} px, {m['dtype']}"
    raise ValueError("Could not read this file. It needs its label: for PDS3 the header is inside the .IMG (or a separate .LBL - put that "
                     "path in LABEL_FILE); for PDS4 upload the .xml next to the .IMG (and run Cell I0 first).")

# ---- get the file ----
src = IMG_INPUT.strip().strip('"').strip("'")
os.makedirs('/content/luminar_work/converted', exist_ok=True)
if not src:
    from google.colab import files
    up = files.upload()
    if not up: raise ValueError("Nothing uploaded.")
    names = list(up)
    src = next((n for n in names if n.lower().endswith('.img')), names[0])
    lab = next((n for n in names if n.lower().endswith(('.lbl', '.xml'))), '')
    if lab and not LABEL_FILE: LABEL_FILE = lab
elif re.match(r'https?://', src, re.I):
    if '_download' not in globals(): raise NameError("Run Cell I1 first (it defines the link downloader), or give a path instead of a link.")
    src = _download(src, '/content/luminar_work/converted')[0]
    if os.path.splitext(src)[1] == '': os.rename(src, src + '.IMG'); src += '.IMG'
if not os.path.exists(src): raise FileNotFoundError(src)

arr, nodata, what = _open_any(src, LABEL_FILE.strip() or None)
print("Opened:", what)
r0, r1 = 0, arr.shape[0]
if ROWS.strip():
    a, b = ROWS.split(':'); r0, r1 = int(a or 0), int(b or arr.shape[0])
H, W = r1 - r0, arr.shape[1]
step = max(1, int(np.ceil(max(H, W) / MAX_SIDE)))
a = np.asarray(arr[r0:r1:step, ::step], dtype=np.float32)
valid = np.isfinite(a) & (a != (nodata if nodata is not None else np.nan))
if arr.dtype != np.uint8: valid &= (a > 0)
lo, hi = np.percentile(a[valid], (1, 99)) if valid.any() else (0, 1)
u8 = (np.clip((a - lo) / max(hi - lo, 1e-9), 0, 1) * 255).astype(np.uint8)
out = os.path.join('/content/luminar_work/converted', os.path.splitext(os.path.basename(src))[0] + '.png')
cv2.imwrite(out, u8)
print(f"Saved {out}  ({u8.shape[1]} x {u8.shape[0]} px, every {step}th pixel kept)")
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 6 * u8.shape[0] / max(u8.shape[1], 1) if u8.shape[0] < 3 * u8.shape[1] else 9)); plt.imshow(u8, cmap='gray'); plt.axis('off'); plt.show()
except Exception: pass
print("\nNext: in the Image-upload panel of Cell I1/I2 choose this PNG, OR paste its path in the Zip box:", out)
try:
    from google.colab import files as _f; _f.download(out)
except Exception: pass

In [ ]:
# ===== CELL I1: SOURCE image (the Chandrayaan-2 image to be aligned) =====
# Run this cell once. A panel appears: choose  [Zip from a Google Drive link]  or  [Image upload], give the link / pick the file,
# then press START. Wait for "Ready" (if a zip holds several strips, choose one in the dropdown), then run the next cell.
# Changing things in the panel is always safe - nothing runs until you press Start.

import urllib.request, urllib.parse, subprocess, sys

def _sniff_ext(path):
    with open(path, 'rb') as f: h = f.read(16)
    if h[:4] == b'PK\x03\x04': return '.zip'
    if h[:8] == b'\x89PNG\r\n\x1a\n': return '.png'
    if h[:3] == b'\xff\xd8\xff': return '.jpg'
    if h[:4] in (b'II*\x00', b'MM\x00*'): return '.tif'
    if h[:4] == b'RIFF' and h[8:12] == b'WEBP': return '.webp'
    if h[:5].lower() in (b'<!doc', b'<html'):
        raise ValueError(f"'{os.path.basename(path)}' is a web page, not a file. The link is not public (share it as 'Anyone with the link') or needs a login.")
    return ''

def _fix_ext(p):
    if os.path.splitext(p)[1].lower() in ('.zip', '.img', '.xml') + IMG_EXT:
        return p
    ne = _sniff_ext(p)
    if ne:
        q = p + ne; os.rename(p, q); print(f"  recognised {ne[1:]} from its contents -> {os.path.basename(q)}"); return q
    return p

def _download(url, d):
    if 'drive.google.com' in url or 'docs.google.com' in url:
        try:
            import gdown
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True); import gdown
        if '/folders/' in url:
            print("Downloading the shared Drive folder ...")
            out = gdown.download_folder(url=url, output=d, quiet=False, use_cookies=False)
            if not out: raise RuntimeError("Could not read that Drive folder. Share it as 'Anyone with the link'.")
            return list(out)
        print("Downloading from Google Drive ...")
        out = gdown.download(url=url, output=d + os.sep, quiet=False, fuzzy=True)
        if not out:
            raise RuntimeError("Google Drive refused the download. Share the file as 'Anyone with the link' (Viewer), "
                               "check the link points to the FILE (not your My Drive page), and try again later if it says 'quota'.")
        return [out]
    dest = os.path.join(d, os.path.basename(urllib.parse.urlparse(url).path) or 'download')
    print("Downloading", url)
    with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})) as r, open(dest, 'wb') as f:
        shutil.copyfileobj(r, f, 1 << 20)
    return [dest]

def _local(text):
    p = os.path.expanduser(text)
    if os.path.isdir(p):
        fs = sorted(glob.glob(os.path.join(p, '*')), key=os.path.getmtime, reverse=True)
        return [f for f in fs if os.path.isfile(f) and os.path.splitext(f)[1].lower() in ('.zip', '.img') + IMG_EXT]
    if os.path.isfile(p):
        return [p]
    fs = glob.glob(os.path.join('/content', glob.escape(text))) + glob.glob(os.path.join('/content', '*', glob.escape(text)))
    if os.path.isdir('/content/drive/MyDrive'):
        fs += glob.glob('/content/drive/MyDrive/**/' + glob.escape(text), recursive=True)
    fs = [f for f in fs if os.path.isfile(f)]
    if not fs:
        raise FileNotFoundError(f"'{text}' not found. Paste a Drive/https link or a path like /content/file.zip, or use the Image upload option.")
    return fs[:1]

class _NotReady:
    """Placeholder until you press Start in the panel above (Cell I3 then says what is missing)."""
    def __init__(self, role): self.role = role
    @property
    def choice(self):
        raise ValueError(f"The {self.role} is not loaded yet: use the {self.role.upper()} panel above (choose an option, then press Start) and wait until it says Ready.")

SRC_PICK, REF_PICK = _NotReady('source'), _NotReady('reference')

def _upload_items(fu):
    v = fu.value
    pairs = [(n, d['content']) for n, d in v.items()] if isinstance(v, dict) else [(d['name'], d['content']) for d in v]
    return pairs

def make_panel(role, title):
    import ipywidgets as W
    var = 'SRC_PICK' if role == 'source' else 'REF_PICK'
    d = os.path.join(WORK_ROOT, role + '_in'); os.makedirs(d, exist_ok=True)
    mode = W.ToggleButtons(options=['Zip from a Google Drive link', 'Image upload'], value='Zip from a Google Drive link',
                           description='', style={'button_width': '230px'})
    link = W.Text(placeholder="paste the Drive share link here (file shared as 'Anyone with the link' - any account), or a path like /content/x.zip",
                  layout=W.Layout(width='98%'))
    fu = W.FileUpload(accept='.png,.jpg,.jpeg,.tif,.tiff,.webp,.bmp,.img,.xml', multiple=True, description='Choose image(s)')
    go = W.Button(description='Start', button_style='primary', icon='play', layout=W.Layout(width='160px'))
    status, out = W.HTML(), W.Output()
    hint = W.HTML("<small>Zip: big ISSDC zips are slow to upload, so put the zip on Google Drive (any account), share it, paste the link, press Start.<br>"
                  "Image: screenshots / downloaded pictures (png, jpg, tif, webp), or a PDS4 .img together with its .xml. Pick the file(s), then press Start.</small>")
    def show(_=None):
        zipm = mode.value.startswith('Zip')
        link.layout.display = '' if zipm else 'none'
        fu.layout.display = 'none' if zipm else ''
    mode.observe(show, names='value'); show()

    def start(_b):
        go.disabled = True; status.value = '<b>Working ...</b> (a big download can take a few minutes)'
        with out:
            clear_output(wait=True)
            try:
                if mode.value.startswith('Zip'):
                    t = link.value.strip().strip('"').strip("'")
                    if not t:
                        raise ValueError("Paste the Drive link (or a /content/... path) in the box first, then press Start.")
                    paths = _download(t, d) if re.match(r'https?://', t, re.I) else _local(t)
                else:
                    items = _upload_items(fu)
                    if not items:
                        raise ValueError("No file chosen yet: press 'Choose image(s)', pick your file(s), then press Start. "
                                         "(If the chooser does not work, drag the file into Colab's folder panel and use the Zip option with its /content/... path.)")
                    paths = []
                    for n, c in items:
                        p = os.path.join(d, os.path.basename(n))
                        with open(p, 'wb') as f: f.write(bytes(c))
                        paths.append(p)
                paths = [_fix_ext(p) for p in paths]
                picker = Picker(role, list_options(paths))
                globals()[var] = picker
                status.value = f"<b style='color:#15803d'>Ready</b> - {len(picker.options)} candidate(s)." + \
                               (" Choose one in the dropdown below, then continue." if len(picker.options) > 1 else " Continue with the next cell.")
            except Exception as e:
                status.value = "<b style='color:#b91c1c'>Failed</b>"
                print(f"{type(e).__name__}: {e}")
        go.disabled = False
    go.on_click(start)
    display(W.VBox([W.HTML(f"<h4 style='margin:4px 0'>{title}</h4>"), mode, link, fu, W.HBox([go, status]), hint, out]))

make_panel('source', 'SOURCE image (the Chandrayaan-2 image to be aligned)')

In [ ]:
# ===== CELL I2: REFERENCE image (LRO / QuickMap / other mission) =====
# Same panel as the source: choose [Zip from a Google Drive link] or [Image upload], then press START and wait for "Ready".
if 'make_panel' not in globals():
    raise NameError("Run Cell I1 first (it defines the panel this cell uses).")
make_panel('reference', 'REFERENCE image (LRO / QuickMap / other mission)')

In [ ]:
# ===== CELL I3: open both images, detect what they are, and set the trim lines =====
TRIM_MODE            = "Interactive sliders"   #@param ["Interactive sliders", "Auto (textured block)", "Full image", "Manual box"]
SOURCE_MANUAL_BOX    = ""                      #@param {type:"string"}
REFERENCE_MANUAL_BOX = ""                      #@param {type:"string"}
LAT_LIMIT            = 85                      #@param {type:"number"}
# Interactive sliders : two red lines (top/bottom) and two green lines (left/right) - keep only what you want
# Auto                : source = most textured block (skips no-data edges and, for geo-referenced strips, areas
#                       closer to a pole than LAT_LIMIT); reference = the whole picture
# Manual box          : type x0,y0,x1,y1 in pixels of the ORIGINAL image

src_r, ref_r = open_raster(SRC_PICK.choice), open_raster(REF_PICK.choice)
src_geo, ref_geo = make_geo(src_r), make_geo(ref_r)

def _what(role, r, g):
    print(f"{role.upper():10s} {os.path.basename(r.path)}  {r.shape[1]} x {r.shape[0]} px  | {r.kind}" +
          (f"  | {r.res_m:g} m/px" if r.res_m else "") +
          (f"  | sun elevation {r.meta['sun_elev']:.1f} deg" if r.meta and r.meta.get('sun_elev') is not None else "") +
          (f"  | geometry: {g.kind}" if g else "  | no coordinates in the file (plain picture)"))
    if g is not None and g.kind == 'unknown':
        print("           NOTE: projection not stated -> lat/lon are approximate (linear between the label's corners)")
    if r.meta and r.meta.get('sun_elev') is not None and r.meta['sun_elev'] < 20:
        print(f"           NOTE: sun is only {r.meta['sun_elev']:.0f} deg above the horizon -> long shadows, harder to match")
_what('source', src_r, src_geo); _what('reference', ref_r, ref_geo)

print("\n--- SOURCE trim ---")
SRC_SEL = trim_setup(TRIM_MODE, src_r, src_geo, 'source', SOURCE_MANUAL_BOX, LAT_LIMIT)
print("\n--- REFERENCE trim ---")
REF_SEL = trim_setup(TRIM_MODE, ref_r, ref_geo, 'reference', REFERENCE_MANUAL_BOX, LAT_LIMIT)
print("\nAdjust the lines above if you want, then run the next cell.")


In [ ]:
# ===== CELL I4: apply the trim, orient, scale, save -> source_path / reference_path =====
REFERENCE_M_PER_PX = ""      #@param {type:"string"}
DETAIL_RATIO       = 3.0     #@param {type:"number"}
MAX_SIDE           = 2000    #@param {type:"integer"}
SOURCE_NORTH_UP    = True    #@param {type:"boolean"}
# REFERENCE_M_PER_PX : only needed if the reference is a screenshot (QuickMap shows 'RES ... m/px' bottom-left); else leave blank
# DETAIL_RATIO       : the finer image is shrunk until it is at most this many times sharper than the coarser one
#                      (the matcher's scale search covers roughly 0.25x-4x; a 17x gap fails)
# SOURCE_NORTH_UP    : if the source has coordinates, rotate/mirror it so north is up, like QuickMap

ref_res = float(REFERENCE_M_PER_PX) if str(REFERENCE_M_PER_PX).strip() else ref_r.res_m
src_res_in = src_r.res_m or (float(np.mean(src_geo.orientation(src_r.shape[0] / 2, src_r.shape[1] / 2)['km_per_px']) * 1000) if src_geo else None)
tgt_src = tgt_ref = None
if src_res_in and ref_res:
    if src_res_in < ref_res:   tgt_src = ref_res / DETAIL_RATIO        # source is finer -> bring it closer to the reference
    elif ref_res < src_res_in: tgt_ref = src_res_in / DETAIL_RATIO     # reference is finer -> shrink the reference instead
else:
    print(f"Pixel size unknown for: {'source ' if not src_res_in else ''}{'reference' if not ref_res else ''} -> only limiting size to MAX_SIDE. "
          "For a screenshot, type its RES (QuickMap shows it bottom-left) in REFERENCE_M_PER_PX so the scales can be matched.")

S = prepare_image(src_r, SRC_SEL.box, src_geo, 'source', tgt_src, MAX_SIDE, SOURCE_NORTH_UP)
Rf = prepare_image(ref_r, REF_SEL.box, ref_geo, 'reference', tgt_ref, MAX_SIDE, north_up=True)
source_path, reference_path = S['path'], Rf['path']

print(f"SOURCE    kept {S['box']}  -> {S['shape'][1]} x {S['shape'][0]} px" + (f" at ~{S['res_m_out']:.0f} m/px" if S['res_m_out'] else "") +
      (f"  (shrunk x{S['scale']:.2f})" if S['scale'] < 0.999 else "") +
      ("  | rotated/mirrored to north-up" if S['north_up'] else ""))
if S['orientation'] is not None:
    o = S['orientation']
    print(f"          north was {o['north_deg']:.0f} deg clockwise from the top of the original" + (" and the image was MIRRORED" if o['mirrored'] else ""))
    describe_location(src_geo, S['box'])
print(f"REFERENCE kept {Rf['box']}  -> {Rf['shape'][1]} x {Rf['shape'][0]} px" + (f" at ~{Rf['res_m_out']:.0f} m/px" if Rf['res_m_out'] else "") +
      (f"  (shrunk x{Rf['scale']:.2f})" if Rf['scale'] < 0.999 else ""))
if S['res_m_out'] and (Rf['res_m_out'] or ref_res):
    print(f"expected size ratio source:reference = {(Rf['res_m_out'] or ref_res) / S['res_m_out']:.2f} (1 = same pixel size)")

meta = dict(source=dict(res_m=S['res_m_out'], geo=S['geo_kind'], north_up=S['north_up'], sun_elev=(src_r.meta or {}).get('sun_elev')),
            reference=dict(res_m=Rf['res_m_out'] or ref_res, geo=Rf['geo_kind'], sun_elev=(ref_r.meta or {}).get('sun_elev')))

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
for a, p, t in ((ax[0], source_path, "SOURCE (what will be aligned)"), (ax[1], reference_path, "REFERENCE")):
    a.imshow(cv2.imread(p, cv2.IMREAD_GRAYSCALE), cmap='gray'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()
print("source_path =", source_path, "| reference_path =", reference_path, "\nLook at the two pictures: you should be able to spot 3+ of the same craters/ridges before running Cell 8c.")


In [ ]:
# Cell 3 — helpers: image I/O, coordinate maps, small geometry utils
def read_image(path):
    """Read PNG/JPG/TIF (8/16-bit, gray/colour/alpha) -> 8-bit BGR."""
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(f"OpenCV couldn't read '{path}'.")
    if im.ndim == 3 and im.shape[2] == 4:
        im = im[..., :3]
    if im.dtype != np.uint8:
        lo, hi = np.percentile(im, (0.5, 99.5))
        im = np.clip((im.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1)
        im = (im * 255).astype(np.uint8)
    if im.ndim == 2:
        im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    return im

def to_gray(bgr, clahe=True):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    if clahe:
        g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(g)
    return g

def resize_by(img, s):
    """Resize by factor s. Returns (image, M) where M maps ORIGINAL pixel coords -> resized coords
    (exact, using the integer output size and OpenCV's pixel-centre convention)."""
    h, w = img.shape[:2]
    nw, nh = max(1, int(round(w * s))), max(1, int(round(h * s)))
    if (nw, nh) == (w, h):
        return img, np.eye(3)
    out = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
    sx, sy = nw / w, nh / h
    M = np.array([[sx, 0, 0.5 * sx - 0.5], [0, sy, 0.5 * sy - 0.5], [0, 0, 1.0]])
    return out, M

def apply_H(H, pts):
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    p = np.hstack([pts, np.ones((len(pts), 1))]) @ H.T
    return p[:, :2] / p[:, 2:3]

def to_original(M, pts):
    """Map points from resized coords back to original coords."""
    return apply_H(np.linalg.inv(M), pts)

def auto_trim(img, max_frac=0.15, tol_std=6.0, passes=3):
    """Find the usable image area: trims border bands made of flat / near-black / near-white rows or columns
    (screenshot margins, text headers, solid frames). Inside each edge band (<= max_frac of that side) it trims up to
    the innermost junk line, provided >= 60% of the band lines up to there are junk. Returns (x0, y0, x1, y1).
    Corner badges (e.g. a 'Licensable' tag) can't be found this way -> pass src_crop / ref_crop."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    def junk(v):
        flat = np.mean(np.abs(v - np.median(v)) <= 6) >= 0.75          # one flat colour (margin / text on solid bg)
        return bool(v.std() < tol_std or flat or np.mean((v < 12) | (v > 243)) > 0.92)
    def band(get, limit):
        best = cnt = 0
        for i in range(limit):
            j = junk(get(i)); cnt += j
            if j and cnt / (i + 1) >= 0.6:
                best = i + 1
        return best
    y0, y1, x0, x1 = 0, g.shape[0], 0, g.shape[1]
    for _ in range(passes):
        ly, lx = int(max_frac * (y1 - y0)), int(max_frac * (x1 - x0))
        ny0 = band(lambda i: g[y0 + i, x0:x1], ly); ny1 = band(lambda i: g[y1 - 1 - i, x0:x1], ly)
        nx0 = band(lambda i: g[y0:y1, x0 + i], lx); nx1 = band(lambda i: g[y0:y1, x1 - 1 - i], lx)
        if not (ny0 or ny1 or nx0 or nx1):
            break
        y0, y1, x0, x1 = y0 + ny0, y1 - ny1, x0 + nx0, x1 - nx1
    return (x0, y0, x1, y1)

def show_for_cropping(path):
    """Display an image with a pixel grid so you can read off a crop box (x0, y0, x1, y1)."""
    im = read_image(path)
    plt.figure(figsize=(9, max(3, 9 * im.shape[0] / im.shape[1])))
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); plt.grid(True, alpha=0.5)
    plt.title(f"{Path(path).name}  {im.shape[1]}x{im.shape[0]}  — read crop box (x0,y0,x1,y1) from the axes"); plt.show()

In [ ]:
# Cell 4 — Stage 1: multi-scale SIFT sweep (RootSIFT)
def rootsift(des):
    des = des / (des.sum(axis=1, keepdims=True) + 1e-7)
    return np.sqrt(des).astype(np.float32)

class SiftMatcher:
    def __init__(self, nfeatures, ratio):
        self.sift = cv2.SIFT_create(nfeatures=nfeatures)
        self.bf = cv2.BFMatcher(cv2.NORM_L2)
        self.ratio = ratio

    def feats(self, gray):
        kp, des = self.sift.detectAndCompute(gray, None)
        if des is None or len(kp) == 0:
            return np.zeros((0, 2), np.float32), None
        return np.float32([k.pt for k in kp]), rootsift(des)

    def match(self, f0, f1):
        (p0, d0), (p1, d1) = f0, f1
        empty = np.zeros((0, 2), np.float32), np.zeros((0, 2), np.float32), np.zeros(0)
        if d0 is None or d1 is None or len(d0) < 2 or len(d1) < 2:
            return empty
        knn = self.bf.knnMatch(d0, d1, k=2)
        q, t, c = [], [], []
        for pr in knn:
            if len(pr) == 2 and pr[0].distance < self.ratio * pr[1].distance:
                q.append(pr[0].queryIdx); t.append(pr[0].trainIdx)
                c.append(1.0 - pr[0].distance / max(pr[1].distance, 1e-9))
        if not q:
            return empty
        return p0[q], p1[t], np.array(c)

def make_views(gray, tilts, rot_base=72.0, max_views=24, max_side=3000):
    """ASIFT-style viewpoint simulation: affine tilt (compress along a rotated axis by 1/t) for several tilts/rotations.
    Returns [(view, M, mask)] with M mapping `gray` coords -> view coords (first entry is the untouched image)."""
    h, w = gray.shape[:2]
    views = [(gray, np.eye(3), None)]
    for t in tilts:
        if t <= 1.0:
            continue
        for phi in np.arange(0, 180, rot_base / t):
            R = cv2.getRotationMatrix2D((w / 2, h / 2), float(phi), 1.0)
            M = np.diag([1.0 / t, 1.0, 1.0]) @ np.vstack([R, [0, 0, 1]])
            cc = apply_H(M, np.float64([[0, 0], [w, 0], [w, h], [0, h]]))
            mn = cc.min(0)
            M = np.array([[1, 0, -mn[0]], [0, 1, -mn[1]], [0, 0, 1.0]]) @ M
            size = tuple(np.ceil(cc.max(0) - mn).astype(int))
            if max(size) > max_side or min(size) < 24:
                continue
            view = cv2.warpPerspective(gray, M, size, flags=cv2.INTER_LINEAR)
            mask = cv2.erode(cv2.warpPerspective(np.full((h, w), 255, np.uint8), M, size, flags=cv2.INTER_NEAREST),
                             np.ones((9, 9), np.uint8))
            views.append((view, M, mask))
    if len(views) > max_views:
        keep = np.unique(np.linspace(1, len(views) - 1, max_views - 1).astype(int))
        views = [views[0]] + [views[i] for i in keep]
    return views

def sift_match_views(sm, views, f_other):
    """SIFT on every view (points back-projected to the un-warped image's coords), ratio-test against `f_other`.
    Returns (pts_in_views_image, pts_in_other, conf)."""
    P, D = [], []
    for v, M, mask in views:
        kp, des = sm.sift.detectAndCompute(v, mask)
        if des is None or len(kp) == 0:
            continue
        pts = np.float32([k.pt for k in kp])
        P.append(apply_H(np.linalg.inv(M), pts).astype(np.float32)); D.append(rootsift(des))
    if not P:
        return np.zeros((0, 2), np.float32), np.zeros((0, 2), np.float32), np.zeros(0)
    return sm.match((np.vstack(P), np.vstack(D)), f_other)

def sift_vp_candidates(sm, g0, g1, cfg, tilts, rot_base=None):
    """Viewpoint-simulated SIFT candidates, source->ref (and ref->source if cfg['vp_both_directions'])."""
    rb = rot_base or cfg['rot_base']
    f0 = sm.feats(g0); f1 = sm.feats(g1)
    out = [sift_match_views(sm, make_views(g0, tilts, rb, cfg['max_views']), f1)]
    if cfg['vp_both_directions']:
        b1, b0, c = sift_match_views(sm, make_views(g1, tilts, rb, cfg['max_views']), f0)
        out.append((b0, b1, c))
    p0 = np.vstack([o[0] for o in out]); p1 = np.vstack([o[1] for o in out]); c = np.concatenate([o[2] for o in out])
    return p0, p1, c

def scale_sweep(src_w, ref_w, cfg, vp=False):
    """Find the scale factor of the source (relative to the reference) with the most geometric consensus.
    vp=True additionally simulates viewpoints (slower; used when the plain sweep is not confident)."""
    sm = SiftMatcher(cfg['sift_nfeatures'], cfg['sift_ratio'])
    f_ref = sm.feats(to_gray(ref_w, cfg['clahe']))
    rows = []
    for s in (cfg['vp_scales'] if vp else cfg['scales']):
        h, w = src_w.shape[:2]
        if h * s * w * s > cfg['max_work_pixels'] or min(h, w) * s < 32:
            continue
        im, _ = resize_by(src_w, s)
        g = to_gray(im, cfg['clahe'])
        if vp:
            p0, p1, _ = sift_match_views(sm, make_views(g, cfg['sweep_tilts'], 90.0, 12), f_ref)
        else:
            p0, p1, _ = sm.match(sm.feats(g), f_ref)
        n_in = 0
        if len(p0) >= 4:
            H, m = cv2.findHomography(p0, p1, cv2.USAC_MAGSAC, cfg['sift_sweep_px'])
            if H is not None and m is not None:
                n_in = int(m.sum())
        rows.append((s, len(p0), n_in))
        log(f"   scale {s:>5}x  ratio-test matches {len(p0):>5}  MAGSAC inliers {n_in:>4}")
    if not rows:
        return 1.0, False, rows
    best = max(rows, key=lambda r: (r[2], -abs(np.log(r[0]))))   # tie -> closest to 1.0x
    confident = best[2] >= cfg['min_scale_inliers']
    return (best[0] if confident else 1.0), confident, rows

In [ ]:
# Cell 5 — Stage 2/3: SuperPoint + LightGlue (with graceful fallback) and candidate pooling
_MODELS = {}

def deep_match_views(views, g_other, cfg, swap=False):
    """SuperPoint + LightGlue between each view of one image and the other image. Points are back-projected to the
    un-warped image's coordinates. swap=True means `views` belong to the REFERENCE (pairs are swapped back).
    Returns ((p0, p1, conf), info) or (None, reason) if torch/lightglue are unavailable."""
    try:
        import torch
        from lightglue import LightGlue, SuperPoint
        from lightglue.utils import rbd
    except Exception as e:
        return None, f"SuperPoint/LightGlue unavailable ({type(e).__name__}: {e})"
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if 'ext' not in _MODELS:
        _MODELS['ext'] = SuperPoint(max_num_keypoints=cfg['sp_max_kp']).eval().to(dev)
        _MODELS['lg'] = LightGlue(features='superpoint', filter_threshold=cfg['lg_filter_threshold']).eval().to(dev)
    ext, lg = _MODELS['ext'], _MODELS['lg']
    t = lambda g: (torch.from_numpy(np.ascontiguousarray(g)).float()[None, None] / 255.0).to(dev)
    A, B, C = [], [], []
    n_raw = n_kp_v = 0
    with torch.no_grad():
        fo = ext.extract(t(g_other))
        n_kp_o = int(fo['keypoints'].shape[1])
        for v, M, _mask in views:
            try:
                fv = ext.extract(t(v))
                m01 = lg({'image0': fv, 'image1': fo})
            except Exception:
                continue
            fv_, fo_, m_ = rbd(fv), rbd(fo), rbd(m01)
            kv, ko = fv_['keypoints'].detach().cpu().numpy(), fo_['keypoints'].detach().cpu().numpy()
            mt = m_['matches'].detach().cpu().numpy()
            sc = m_['scores'].detach().cpu().numpy() if 'scores' in m_ else np.ones(len(mt))
            n_raw += len(mt); n_kp_v += len(kv)
            if len(mt):
                A.append(apply_H(np.linalg.inv(M), kv[mt[:, 0]])); B.append(ko[mt[:, 1]]); C.append(sc)
    info = dict(n_kp0=n_kp_v, n_kp1=n_kp_o, n_raw=n_raw, n_views=len(views), device=str(dev))
    if not A:
        return (np.zeros((0, 2)), np.zeros((0, 2)), np.zeros(0)), info
    a, b, c = np.vstack(A), np.vstack(B), np.concatenate(C)
    return ((b, a, c) if swap else (a, b, c)), info

def merge_candidates(parts, dedupe_px):
    """parts: list of (p0, p1, conf, label) in ORIGINAL coords. Concatenate and drop near-duplicates (earlier wins)."""
    parts = [p for p in parts if len(p[0])]
    if not parts:
        return np.zeros((0, 2)), np.zeros((0, 2)), np.zeros(0), np.array([], dtype=object)
    p0 = np.vstack([p[0] for p in parts]); p1 = np.vstack([p[1] for p in parts])
    conf = np.concatenate([p[2] for p in parts])
    lab = np.concatenate([[p[3]] * len(p[0]) for p in parts]).astype(object)
    keep = np.ones(len(p0), bool)
    if len(p0) > 1:
        for i, j in cKDTree(np.hstack([p0, p1])).query_pairs(dedupe_px):
            keep[max(i, j)] = False
    return p0[keep], p1[keep], conf[keep], lab[keep]

In [ ]:
# Cell 6 — Stage 4: Lunar Structural Graph (triangle-consistency) + uniform selection
def lsg_scores(p0, p1, k, tol, min_area_ratio):
    """Per-match score = fraction of local triangles (from the k-NN of both images) whose side-ratios agree
    between source and reference within `tol`. Near-degenerate (sliver) triangles are ignored."""
    n = len(p0)
    good, tot = np.zeros(n), np.zeros(n)
    if n < 4:
        return np.zeros(n), tot
    kk = min(k + 1, n)
    nb0 = cKDTree(p0).query(p0, k=kk)[1]
    nb1 = cKDTree(p1).query(p1, k=kk)[1]
    for i in range(n):
        nb = np.unique(np.concatenate([nb0[i], nb1[i]]))
        nb = nb[nb != i]
        if len(nb) < 2:
            continue
        a, b = np.triu_indices(len(nb), 1)
        j, q = nb[a], nb[b]
        sides, areas = [], []
        for P in (p0, p1):
            dij = np.linalg.norm(P[i] - P[j], axis=1)
            diq = np.linalg.norm(P[i] - P[q], axis=1)
            djq = np.linalg.norm(P[j] - P[q], axis=1)
            cross = np.abs((P[j, 0] - P[i, 0]) * (P[q, 1] - P[i, 1]) - (P[j, 1] - P[i, 1]) * (P[q, 0] - P[i, 0]))
            mx = np.maximum(np.maximum(dij, diq), djq)
            sides.append((dij, diq, djq)); areas.append(0.5 * cross / np.maximum(mx ** 2, 1e-9))
        ok = (areas[0] >= min_area_ratio) & (areas[1] >= min_area_ratio)
        ok &= np.minimum.reduce([*sides[0], *sides[1]]) > 1e-3
        if not ok.any():
            continue
        (a0, b0, c0), (a1, b1, c1) = [tuple(x[ok] for x in sd) for sd in sides]
        rs = np.stack([a0 / b0, b0 / c0, c0 / a0], 1)
        rr = np.stack([a1 / b1, b1 / c1, c1 / a1], 1)
        cons = (np.mean(np.abs(rs - rr) / (rs + 1e-9), axis=1) < tol).astype(float)
        jj, qq = j[ok], q[ok]
        for idx in (np.full(len(cons), i), jj, qq):
            np.add.at(good, idx, cons); np.add.at(tot, idx, 1.0)
    return np.divide(good, tot, out=np.zeros(n), where=tot > 0), tot

def lsg_filter(p0, p1, cfg):
    """Iterative pruning: drop the worst-scoring matches (max 30% per round), re-score, repeat.
    Returns dict(keep_idx, score, ok, iters). If it cannot keep `lsg_min_keep` matches the filter is declared
    FAILED (ok=False) and ALL matches are passed through — and the report says so (no silent fallback)."""
    n = len(p0)
    idx = np.arange(n)
    last_score = np.zeros(n)
    sc, it = np.zeros(0), 0
    for it in range(1, cfg['lsg_iters'] + 1):
        sc, _ = lsg_scores(p0[idx], p1[idx], cfg['lsg_k'], cfg['lsg_tol'], cfg['lsg_min_area_ratio'])
        last_score[idx] = sc
        bad = np.where(sc < cfg['lsg_min_score'])[0]
        if len(bad) == 0:
            break
        bad = bad[np.argsort(sc[bad])][:max(1, int(0.3 * len(idx)))]
        keep = np.ones(len(idx), bool); keep[bad] = False
        if keep.sum() < cfg['lsg_min_keep']:
            return dict(keep_idx=np.arange(n), score=last_score, ok=False, iters=it)
        idx = idx[keep]
    return dict(keep_idx=idx, score=last_score, ok=True, iters=it)

def select_uniform(p_ref, score, ref_shape, grid, max_per_cell):
    h, w = ref_shape[:2]
    cells = {}
    for i, (x, y) in enumerate(p_ref):
        cells.setdefault((int(min(x // (w / grid), grid - 1)), int(min(y // (h / grid), grid - 1))), []).append(i)
    keep = []
    for ids in cells.values():
        ids.sort(key=lambda i: score[i], reverse=True)
        keep.extend(ids[:max_per_cell])
    return np.array(sorted(keep), dtype=int)

In [ ]:

# Cell 7 — Stage 5/6: MAGSAC++ + Huber refinement, honest metrics, verdict
def hartley(p):
    c = p.mean(0)
    d = np.sqrt(((p - c) ** 2).sum(1)).mean()
    s = np.sqrt(2) / max(d, 1e-9)
    return np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1.0]])

def refine_huber(H0, p0, p1, fscale):
    """Huber-loss non-linear refinement of H on (normalised) coordinates, 8 DoF (h33 fixed = 1)."""
    T0, T1 = hartley(p0), hartley(p1)
    T1i = np.linalg.inv(T1)
    Hn0 = T1 @ H0 @ np.linalg.inv(T0)
    Hn0 = Hn0 / Hn0[2, 2]
    def res(h):
        H = T1i @ np.append(h, 1.0).reshape(3, 3) @ T0
        return (apply_H(H, p0) - p1).ravel()
    r = least_squares(res, Hn0.ravel()[:8], loss='huber', f_scale=fscale, method='trf', x_scale='jac', max_nfev=300)
    H = T1i @ np.append(r.x, 1.0).reshape(3, 3) @ T0
    return H / H[2, 2], r

def huber_cost(H, p0, p1, f):
    e = np.linalg.norm(apply_H(H, p0) - p1, axis=1)
    return float(np.sum(np.where(e <= f, 0.5 * e ** 2, f * (e - 0.5 * f))))

def check_homography(H, src_shape):
    h, w = src_shape[:2]
    corners = np.float64([[0, 0], [w, 0], [w, h], [0, h]])
    p = np.hstack([corners, np.ones((4, 1))]) @ H.T
    if not np.all(np.isfinite(p)) or np.any(p[:, 2] <= 1e-9):
        return False, "source footprint crosses the plane at infinity", None
    q = p[:, :2] / p[:, 2:3]
    if np.abs(q).max() > 1e6:
        return False, "absurd footprint size", None
    if not cv2.isContourConvex(q.astype(np.float32).reshape(-1, 1, 2)):
        return False, "source footprint is not a convex quadrilateral", q
    if cv2.contourArea(q.astype(np.float32)) < 16:
        return False, "degenerate footprint", q
    return True, "ok", q

def crossval_rmse(p0, p1, folds, seed=0):
    """Held-out RMSE: fit on (folds-1)/folds of the inliers, test on the rest. Exposes over-fitting that the
    in-sample RMSE hides when the inlier set is small."""
    n = len(p0)
    if n < 10:
        return float('nan')
    perm = np.random.default_rng(seed).permutation(n)
    errs = []
    for f in range(min(folds, n // 2)):
        test = perm[f::folds]
        train = np.setdiff1d(perm, test)
        if len(train) < 8:
            continue
        H, _ = cv2.findHomography(p0[train], p1[train], 0)
        if H is not None:
            errs.append(np.linalg.norm(apply_H(H, p0[test]) - p1[test], axis=1))
    return float(np.sqrt(np.mean(np.concatenate(errs) ** 2))) if errs else float('nan')

def uniformity(pts, footprint_mask, grid=4):
    """Spatial uniformity of the inliers over the actual overlap region."""
    out = dict(coverage=0.0, hull_ratio=0.0, clark_evans=float('nan'), score=0.0)
    ys, xs = np.nonzero(footprint_mask)
    if len(xs) == 0 or len(pts) < 3:
        return out
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    gx = np.linspace(x0, x1, grid + 1); gy = np.linspace(y0, y1, grid + 1)
    valid = occ = 0
    for i in range(grid):
        for j in range(grid):
            cell = footprint_mask[int(gy[j]):int(gy[j + 1]), int(gx[i]):int(gx[i + 1])]
            if cell.size and cell.mean() >= 0.25:
                valid += 1
                inside = (pts[:, 0] >= gx[i]) & (pts[:, 0] < gx[i + 1]) & (pts[:, 1] >= gy[j]) & (pts[:, 1] < gy[j + 1])
                occ += int(inside.any())
    area = float(footprint_mask.sum())
    try:
        hull = min(1.0, ConvexHull(pts).volume / area)
    except Exception:
        hull = 0.0
    nn = cKDTree(pts).query(pts, k=2)[0][:, 1].mean()
    out.update(coverage=occ / max(valid, 1), hull_ratio=hull,
               clark_evans=float(nn / (0.5 * np.sqrt(area / len(pts)))))
    out['score'] = 0.5 * out['coverage'] + 0.5 * out['hull_ratio']
    return out

def occupancy(pts, shape, grid=4):
    """Fraction of grid x grid cells (over `shape`) that contain at least one point."""
    h, w = shape[:2]
    cx = np.clip((pts[:, 0] / w * grid).astype(int), 0, grid - 1); cy = np.clip((pts[:, 1] / h * grid).astype(int), 0, grid - 1)
    return len(set(zip(cx.tolist(), cy.tolist()))) / grid ** 2

def cov_score(n_in, cov):
    """Model score: many inliers AND spread over the image. A tight, self-consistent cluster of look-alike texture
    can win a plain inlier count; it cannot win this."""
    return n_in * (0.25 + 0.75 * cov)

def spread_ransac(P0, P1, src_shape, th, iters, grid=4, seed=0):
    """RANSAC whose 4-point samples are drawn from DIFFERENT cells of the source image (well-conditioned, global
    hypotheses), scored by cov_score. Returns (H, inlier_mask) or None."""
    n = len(P0)
    h, w = src_shape[:2]
    cell = (np.clip((P0[:, 0] / w * grid).astype(int), 0, grid - 1) * grid + np.clip((P0[:, 1] / h * grid).astype(int), 0, grid - 1))
    cells = [np.where(cell == c)[0] for c in range(grid * grid)]
    cells = [c for c in cells if len(c)]
    if len(cells) < 4 or n < 8:
        return None
    rng = np.random.default_rng(seed)
    A, B = P0.astype(np.float64), P1.astype(np.float64)
    best, best_s = None, -1.0
    for _ in range(iters):
        cs = rng.choice(len(cells), 4, replace=False)
        idx = [int(rng.choice(cells[c])) for c in cs]
        try:
            H = cv2.getPerspectiveTransform(A[idx].astype(np.float32), B[idx].astype(np.float32))
        except cv2.error:
            continue
        if not np.isfinite(H).all():
            continue
        m = np.linalg.norm(apply_H(H, A) - B, axis=1) <= th
        k = int(m.sum())
        if k < 6:
            continue
        sc = cov_score(k, len(np.unique(cell[m])) / grid ** 2)
        if sc > best_s and check_homography(H, src_shape)[0]:
            best, best_s = (H, m), sc
    if best is None:
        return None
    H, m = best
    Hl, _ = cv2.findHomography(A[m], B[m], 0)                      # LM polish on the consensus set
    if Hl is not None and check_homography(Hl, src_shape)[0]:
        H = Hl; m = np.linalg.norm(apply_H(H, A) - B, axis=1) <= th
    return H, m

def refine_from_seed(H0, P0, P1, ths, w_pts=None):
    """Iteratively re-fit H from a seed: inliers at each (shrinking) threshold, LM fit. Returns H."""
    H = H0
    for th in ths:
        m = np.linalg.norm(apply_H(H, P0) - P1, axis=1) <= th
        if m.sum() < 6:
            continue
        Hn, _ = cv2.findHomography(P0[m].astype(np.float64), P1[m].astype(np.float64), 0)
        if Hn is not None:
            H = Hn
    return H

def _sample_field(field, step, xs, ys):
    """Bilinear lookup in a coarse (gh, gw, 2) field sampled every `step` px; xs, ys are pixel coords."""
    mx = (np.asarray(xs, np.float32) / step); my = (np.asarray(ys, np.float32) / step)
    return cv2.remap(field, mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

def nonrigid_fit(H, P0, P1, ref_shape, cfg):
    """Thin-plate-spline residual field on top of the global homography (models relief / residual parallax).
    Accepted only if HELD-OUT error improves by >= (1 - nr_gain). Returns dict or None."""
    from scipy.interpolate import RBFInterpolator
    n = len(P0)
    if n < cfg['nr_min_pts']:
        return None
    perm = np.random.default_rng(1).permutation(n)
    folds = min(cfg['cv_folds'], n // 3)
    e_h, e_n = [], []
    for f in range(folds):
        te = perm[f::folds]; tr = np.setdiff1d(perm, te)
        Hf, _ = cv2.findHomography(P0[tr], P1[tr], 0)
        if Hf is None:
            continue
        btr, bte = apply_H(Hf, P0[tr]), apply_H(Hf, P0[te])
        try:
            rbf = RBFInterpolator(btr, P1[tr] - btr, kernel='thin_plate_spline', smoothing=cfg['nr_smoothing'])
            pred = bte + rbf(bte)
        except Exception:
            continue
        e_h.append(np.linalg.norm(bte - P1[te], axis=1)); e_n.append(np.linalg.norm(pred - P1[te], axis=1))
    if not e_h:
        return None
    cv_h = float(np.sqrt(np.mean(np.concatenate(e_h) ** 2))); cv_n = float(np.sqrt(np.mean(np.concatenate(e_n) ** 2)))
    base = apply_H(H, P0)
    rbf = RBFInterpolator(base, P1 - base, kernel='thin_plate_spline', smoothing=cfg['nr_smoothing'])
    rh, rw = ref_shape[:2]
    step = float(max(8.0, np.hypot(rw, rh) / 150))
    gx, gy = np.arange(0, rw + step, step), np.arange(0, rh + step, step)
    G = np.stack(np.meshgrid(gx, gy), -1).reshape(-1, 2)
    d = rbf(G)
    dist = cKDTree(base).query(G)[0]
    d *= np.clip(1 - dist / (cfg['nr_fade'] * np.hypot(rw, rh)), 0, 1)[:, None]          # fade away from the evidence
    d = np.clip(d, -3 * np.abs(P1 - base).max(), 3 * np.abs(P1 - base).max())
    field = d.reshape(len(gy), len(gx), 2).astype(np.float32)
    return dict(field=field, step=step, cv_h=cv_h, cv_nr=cv_n, n=n, use=bool(cv_n < cfg['nr_gain'] * cv_h))

def warp_src(src, H, ref_shape, nr=None, interp=cv2.INTER_CUBIC, fill=0):
    """Warp src into the reference frame by H (optionally + TPS field). Returns image."""
    rh, rw = ref_shape[:2]
    if nr is None or not nr['use']:
        return cv2.warpPerspective(src, H, (rw, rh), flags=interp, borderValue=fill)
    qx, qy = np.meshgrid(np.arange(rw, dtype=np.float32), np.arange(rh, dtype=np.float32))
    d = _sample_field(nr['field'], nr['step'], qx, qy)
    d = _sample_field(nr['field'], nr['step'], qx - d[..., 0], qy - d[..., 1])      # one fixed-point iteration
    bx, by = qx - d[..., 0], qy - d[..., 1]
    Hi = np.linalg.inv(H)
    den = Hi[2, 0] * bx + Hi[2, 1] * by + Hi[2, 2]
    mx = ((Hi[0, 0] * bx + Hi[0, 1] * by + Hi[0, 2]) / den).astype(np.float32)
    my = ((Hi[1, 0] * bx + Hi[1, 1] * by + Hi[1, 2]) / den).astype(np.float32)
    return cv2.remap(src, mx, my, interp, borderMode=cv2.BORDER_CONSTANT, borderValue=fill)

def image_quality(src, ref, H, nr=None):
    rh, rw = ref.shape[:2]
    reg = warp_src(src, H, ref.shape, nr)
    valid = warp_src(np.full(src.shape[:2], 255, np.uint8), H, ref.shape, nr, interp=cv2.INTER_NEAREST) > 0
    valid = cv2.erode(valid.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
    gr, gg = cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY), cv2.cvtColor(reg, cv2.COLOR_BGR2GRAY)
    s_val = n_val = float('nan')
    if valid.sum() > 1000 and min(rh, rw) >= 7:
        _, smap = ssim(gr, gg, data_range=255, full=True)
        s_val = float(smap[valid].mean())
        a, b = gr[valid].astype(np.float64), gg[valid].astype(np.float64)
        if a.std() > 0 and b.std() > 0:
            n_val = float(np.corrcoef(a, b)[0, 1])
    return reg, valid, s_val, n_val

def assess(m):
    """Honest confidence verdict. Never calls a weak fit 'good'."""
    why = []
    if not m['H_ok']:
        return "FAILED", [m['H_msg']]
    n, ratio, cv, tier = m['n_inliers'], m['inlier_ratio'], m['cv_rmse'], m['tier_px']
    if m.get('manual_n'):
        if m['n_auto_inliers'] < 15 or m['uniformity']['score'] < 0.30:
            return "MANUAL", [f"hand-seeded from {m['manual_n']} points; only {m['n_auto_inliers']} automatic matches agree — "
                              f"accuracy is limited by your click precision (residual on your points {m['manual_rmse']:.1f}px), "
                              f"NOT independently verified"]
        why.append(f"manually seeded ({m['manual_n']} points), confirmed by {m['n_auto_inliers']} automatic matches")
    if n < 8:
        return "FAILED", [f"only {n} inliers (<8) — homography is not determined reliably"]
    level = "HIGH"
    order = ["HIGH", "MEDIUM", "LOW"]
    def down(to, reason):
        nonlocal level
        if order.index(to) > order.index(level):
            level = to
        why.append(reason)
    if tier >= 16: down("LOW", f"only reached the {tier:.0f}px accuracy tier")
    elif tier >= 8: down("MEDIUM", f"only reached the {tier:.0f}px accuracy tier")
    elif tier > 2: why.append(f"accuracy tier {tier:.0f}px (strict 2px not reachable with a global homography)")
    if n < 15: down("LOW", f"few inliers ({n})")
    elif n < 30: down("MEDIUM", f"moderate inlier count ({n})")
    if ratio < 0.30: down("LOW", f"low inlier ratio ({ratio:.0%})")
    elif ratio < 0.50: down("MEDIUM", f"inlier ratio {ratio:.0%}")
    if np.isnan(cv): down("LOW", "too few inliers for held-out validation")
    elif cv > 2.0 * max(tier / 2, 1): down("LOW", f"held-out RMSE {cv:.2f}px vs {tier:.0f}px tier (over-fit)")
    elif cv > 1.0 * max(tier / 2, 1): down("MEDIUM", f"held-out RMSE {cv:.2f}px")
    if m['uniformity']['score'] < 0.30: down("LOW", f"inliers cover little of the overlap (uniformity {m['uniformity']['score']:.2f}) — registration may only be valid locally")
    elif m['uniformity']['score'] < 0.45: down("MEDIUM", "uneven inlier coverage")
    if not m['lsg_ok']: down("MEDIUM", "LSG could not verify local geometry (filter bypassed)")
    if not m['scale_confident']: why.append("scale sweep not confident")
    if m.get('nr_used'):
        why.append(f"non-rigid TPS correction used (held-out {m['nr_cv_h']:.1f}px -> {m['nr_cv_nr']:.1f}px)")
    return level, why or ["all checks passed"]

In [ ]:
# Cell 7b — STAGE HOOKS: designed stages, NOT built yet (placeholders with fixed contracts)
# >>> Each hook below is a NO-OP today and is already called by run_pipeline at the right place.
# >>> To install a stage:  HOOKS['NAME'] = my_function   (contracts below; all coordinates are in the CROPPED frames).
HOOKS = {}

HOOK_CONTRACTS = {
    # sensor labels (PDS4 / SPICE): exact pixel-size ratio and/or an initial transform
    'METADATA_ALIGNMENT': "fn(meta, src_shape, ref_shape) -> {'scale': float|None}   (scale = ref px per src px)",
    # >>> PLACEHOLDER: DEM_RENDER_RELIGHT — remove relief and sun: orthorectify with a DEM, relight under each sun angle.
    'DEM_RENDER_RELIGHT': "fn(src_bgr, ref_bgr, meta) -> (src_for_matching, ref_for_matching) pixel-aligned to the crops, or None",
    # crater constellations / Fourier-Mellin phase correlation / any independent global alignment
    'PROPOSE_ALIGNMENTS': "fn(src_bgr, ref_bgr, meta) -> [(name, H_src_to_ref 3x3), ...]",
    # statistical (a-contrario) test replacing hand-set thresholds
    'CHANCE_VERIFICATION': "fn(P0, P1, H, src_shape, ref_shape) -> {'accept': bool, 'nfa': float, 'tol_px': float}",
    # local patch matching on a regular lattice after coarse alignment (uniform by construction)
    'DENSE_TIE_POINTS': "fn(src_bgr, ref_bgr, H) -> (P0 Nx2, P1 Nx2, conf N)",
}

# (label, hook name or None) — None = built. Order = pipeline order.
STAGES = [("Clean and normalize", None), ("Fix scale and viewpoint", None),
          ("Candidate matches (SuperPoint+LightGlue, RootSIFT)", None),
          ("Metadata alignment", 'METADATA_ALIGNMENT'), ("Propose alignments (craters, Fourier-Mellin)", 'PROPOSE_ALIGNMENTS'),
          ("Remove relief and sun (DEM render)", 'DEM_RENDER_RELIGHT'), ("Structural graph (LSG)", None),
          ("Model selection (MAGSAC++, spread-RANSAC)", None), ("Verify against chance (a-contrario)", 'CHANCE_VERIFICATION'),
          ("Refine (Huber, non-rigid)", None), ("Dense tie points", 'DENSE_TIE_POINTS'), ("Report", None)]

def hook_status():
    return {n: ('INSTALLED' if n in HOOKS else 'placeholder') for n in HOOK_CONTRACTS}

def call_hook(name, *args, default=None):
    fn = HOOKS.get(name)
    return default if fn is None else fn(*args)

In [ ]:
# Cell 8a — dashboard + export
PAL = dict(navy="#0F172A", gold="#D97706", green="#15803D", red="#B91C1C", purple="#6D28D9", ice="#F8FAFC",
           HIGH="#15803D", MEDIUM="#D97706", LOW="#B91C1C", FAILED="#7F1D1D", MANUAL="#6D28D9")

def _rgb(img): return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def _tag(ax, text, color):
    ax.text(0.0, 1.05, f" {text} ", transform=ax.transAxes, fontsize=10, fontweight='bold', color='white',
            ha='left', va='center', bbox=dict(boxstyle="round,pad=0.35", facecolor=color, edgecolor="none"))

def checkerboard(a, b, tile=32):
    yy, xx = np.indices(a.shape[:2])
    m = (((yy // tile) + (xx // tile)) % 2 == 0)[..., None]
    return np.where(m, a, b).astype(np.uint8)

def _anaglyph(a, b):
    """a (registered/source) in the RED channel, b (reference) in GREEN+BLUE: aligned = grey, misaligned = red/cyan fringes."""
    return np.dstack([a, b, b])

def build_dashboard(R):
    import textwrap
    src, ref, lvl = R['src'], R['ref'], R['level']
    gray = lambda im: cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    fig = plt.figure(figsize=(13, 28), facecolor="white")
    gs = fig.add_gridspec(7, 2, height_ratios=[1, 1.15, 1, 1, 1, 1, 1.25], hspace=0.22, wspace=0.06)
    ax = fig.add_subplot(gs[0, 0]); ax.imshow(_rgb(src)); ax.axis('off'); _tag(ax, f"SOURCE | {R['src_name']}", PAL['navy'])
    ax = fig.add_subplot(gs[0, 1]); ax.imshow(_rgb(ref)); ax.axis('off'); _tag(ax, f"REFERENCE | {R['ref_name']}", PAL['navy'])

    H0 = max(src.shape[0], ref.shape[0])
    pad = lambda im: cv2.copyMakeBorder(im, 0, H0 - im.shape[0], 0, 0, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    ax = fig.add_subplot(gs[1, :]); ax.imshow(_rgb(np.hstack([pad(src), pad(ref)]))); ax.axis('off')
    off, sel = src.shape[1], R['sel_idx']
    show = sel if len(sel) <= 300 else sel[np.linspace(0, len(sel) - 1, 300).astype(int)]
    for i in show:
        c = PAL['green'] if R['inlier'][i] else PAL['red']
        ax.plot([R['C0'][i, 0], R['C1'][i, 0] + off], [R['C0'][i, 1], R['C1'][i, 1]], color=c, lw=0.9, alpha=0.8)
    _tag(ax, f"FINAL MATCHES — {R['n_inliers']}/{len(sel)} inliers (green) · outliers (red)", PAL['gold'])

    for col, (img, P, name) in enumerate([(src, R['C0'][sel], "SOURCE"), (ref, R['C1'][sel], "REFERENCE")]):
        ax = fig.add_subplot(gs[2, col]); ax.imshow(_rgb(img)); ax.axis('off')
        if len(P) >= 3:
            nb = cKDTree(P).query(P, k=min(5, len(P)))[1]; inl = R['inlier'][sel]; segs, cols = [], []
            for i in range(len(P)):
                for j in nb[i][1:]:
                    segs.append([P[i], P[j]]); cols.append(PAL['green'] if (inl[i] and inl[j]) else PAL['gold'])
            ax.add_collection(LineCollection(segs, colors=cols, linewidths=1.0, alpha=0.85))
            ax.scatter(P[:, 0], P[:, 1], s=26, c=[PAL['green'] if f else PAL['red'] for f in inl], edgecolors='white', linewidths=0.7, zorder=5)
        _tag(ax, f"LUNAR STRUCTURAL GRAPH — {name}", PAL['purple'])

    have = R['reg'] is not None
    if have:
        ax = fig.add_subplot(gs[3, 0]); ax.imshow(_rgb(R['reg'])); ax.axis('off')
        _tag(ax, "REGISTERED OUTPUT" + (" (homography + non-rigid TPS)" if R['nr_used'] else " (homography)"), PAL['green'])
        ax = fig.add_subplot(gs[3, 1]); ax.imshow(_rgb(checkerboard(R['reg'], ref))); ax.axis('off'); _tag(ax, "CHECKERBOARD: registered vs reference", PAL['green'])
        gref, greg = gray(ref), gray(R['reg'])
        naive = cv2.resize(gray(src), (ref.shape[1], ref.shape[0]))
        ax = fig.add_subplot(gs[4, 0]); ax.imshow(_anaglyph(naive, gref)); ax.axis('off'); _tag(ax, "BEFORE: source stretched onto reference (red/cyan)", PAL['red'])
        ax = fig.add_subplot(gs[4, 1]); ax.imshow(_anaglyph(greg, gref)); ax.axis('off'); _tag(ax, "AFTER: registered over reference (grey = aligned)", PAL['green'])
        diff = np.abs(greg.astype(np.float32) - gref.astype(np.float32)); valid = R['valid']
        d_after = float(diff[valid].mean()) if valid.any() else float('nan')
        d_before = float(np.abs(naive.astype(np.float32) - gref.astype(np.float32)).mean())
        dm = np.ma.masked_where(~valid, diff)
        ax = fig.add_subplot(gs[5, 0]); ax.imshow(gref, cmap='gray'); im = ax.imshow(dm, cmap='magma', vmin=0, vmax=120, alpha=0.9); ax.axis('off')
        _tag(ax, f"ABSOLUTE DIFFERENCE (overlap) — mean {d_after:.1f} vs {d_before:.1f} before", PAL['purple'])
    else:
        d_after = d_before = float('nan')
        for r_, c_ in [(3, 0), (3, 1), (4, 0), (4, 1), (5, 0)]:
            ax = fig.add_subplot(gs[r_, c_]); ax.axis('off')
            ax.text(0.5, 0.5, "No valid transform estimated\n(registration not produced)", ha='center', va='center', fontsize=13, color=PAL['red'])
        _tag(ax, "REGISTRATION NOT PRODUCED", PAL['red'])

    ax = fig.add_subplot(gs[5, 1]); ax.imshow(_rgb(ref)); ax.axis('off')
    if R['H'] is not None and len(sel):
        d = apply_H(R['H'], R['C0'][sel]) - R['C1'][sel]
        ax.quiver(R['C1'][sel, 0], R['C1'][sel, 1], d[:, 0], d[:, 1], angles='xy', scale_units='xy', scale=1 / 20,
                  color=[PAL['green'] if f else PAL['red'] for f in R['inlier'][sel]], width=0.004)
    _tag(ax, "RESIDUAL VECTORS (x20 exaggeration)", PAL['navy'])

    # metrics
    ax = fig.add_subplot(gs[6, 0]); ax.axis('off'); _tag(ax, f"METRICS | CONFIDENCE: {lvl}", PAL[lvl])
    u = R['uniformity']
    f = lambda v, fmt: "n/a" if v is None or (isinstance(v, float) and np.isnan(v)) else format(v, fmt)
    lines = [
        f"Matcher            : {R['matcher']}",
        f"Scale found        : {R['scale']:.2f}x ({'confident' if R['scale_confident'] else 'NOT confident -> 1.0x'})",
        f"Candidates / LSG / uniform : {R['n_candidates']} / {R['n_lsg']} / {len(R['sel_idx'])}{'' if R['lsg_ok'] else '  (LSG bypassed!)'}",
        f"Inliers @ {R['tier_px']:.0f}px tier   : {R['n_inliers']}  ({f(100 * R['inlier_ratio'], '.1f')}% of final set)",
        f"Inliers / all cand.: {R['n_inliers_all']} / {R['n_candidates']}",
        "-" * 54,
        f"RMSE (in-sample)   : {f(R['rmse'], '.3f')} px",
        f"RMSE (held-out CV) : {f(R['cv_rmse'], '.3f')} px   <- honest estimate",
        f"Median / P95 / Max : {f(R['med_err'], '.2f')} / {f(R['p95_err'], '.2f')} / {f(R['max_err'], '.2f')} px",
        f"Error / diagonal   : {f(R['err_pct'], '.4f')} %",
        "-" * 54,
        f"Uniformity score   : {f(u['score'], '.2f')}  (grid {u['coverage']:.0%}, hull {u['hull_ratio']:.0%}, Clark-Evans {f(u['clark_evans'], '.2f')})",
        f"SSIM / NCC (overlap): {f(R['ssim'], '.3f')} / {f(R['ncc'], '.3f')}",
        f"Mean abs diff      : {f(d_after, '.1f')} after vs {f(d_before, '.1f')} before",
        f"Model chosen       : {R['model_note']}",
        f"Refinement         : {R['refine_note']}",
        f"Non-rigid (TPS)    : {R['nr_note']}",
        f"Viewpoint sim / crop: {R['vp_note']} / src{R['src_crop']} ref{R['ref_crop']}",
        f"Time               : {R['time']:.1f} s",
        "-" * 54,
        "Verdict notes: " + "; ".join(R['why']),
    ]
    wrapped = []
    for l in lines:
        wrapped += textwrap.wrap(l, 66, subsequent_indent="    ") or [""]
    ax.text(0.02, 0.97, "\n".join(wrapped), transform=ax.transAxes, va='top', ha='left', family='monospace', fontsize=7.2, color="#1E293B")

    # pipeline stages + hook status
    ax = fig.add_subplot(gs[6, 1]); ax.axis('off'); _tag(ax, "PIPELINE STAGES AND RESERVED SLOTS", PAL['navy'])
    hs = hook_status()
    for k, (label, hook) in enumerate(STAGES):
        y = 0.96 - k * 0.075
        if hook is None:
            mark, col, tail = "●", PAL['green'], "built"
        elif hs[hook] == 'INSTALLED':
            mark, col, tail = "●", PAL['purple'], f"installed · {hook}"
        else:
            mark, col, tail = "○", "#64748B", f"PLACEHOLDER · {hook}"
        ax.text(0.02, y, mark, transform=ax.transAxes, color=col, fontsize=10, va='top')
        ax.text(0.07, y, label, transform=ax.transAxes, color="#1E293B", fontsize=8, va='top', fontweight='bold')
        ax.text(0.07, y - 0.032, tail, transform=ax.transAxes, color=col, fontsize=6.8, va='top', family='monospace')
    fig.suptitle("LUMINAR — LUNAR IMAGE REGISTRATION REPORT", fontsize=17, fontweight='bold', color=PAL['navy'], y=0.985)
    fig.subplots_adjust(top=0.955, bottom=0.01)
    return fig

def export_results(R, out_dir):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    cfg = R['cfg']
    if R['reg'] is not None:
        cv2.imwrite(str(out / "registered.png"), R['reg'])
        if R['nr_used']:
            cv2.imwrite(str(out / "registered_homography_only.png"), R['reg_h'])
        cv2.imwrite(str(out / "checkerboard.png"), checkerboard(R['reg'], R['ref']))
    # match-points file — coordinates in ORIGINAL pixel space (x right, y down, pixel centres at integers)
    sel_mask = np.zeros(len(R['C0']), bool); sel_mask[R['sel_idx']] = True
    with open(out / "match_points.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["id", "src_x", "src_y", "ref_x", "ref_y", "detector", "match_conf", "lsg_score",
                    "lsg_kept", "in_final_set", "is_inlier", "residual_px"])
        for i in range(len(R['C0'])):
            r = R['resid'][i]
            w.writerow([i, f"{R['C0'][i,0]+R['src_crop'][0]:.3f}", f"{R['C0'][i,1]+R['src_crop'][1]:.3f}",
                        f"{R['C1'][i,0]+R['ref_crop'][0]:.3f}", f"{R['C1'][i,1]+R['ref_crop'][1]:.3f}",
                        R['lab'][i], f"{R['conf'][i]:.4f}", f"{R['lsg_score'][i]:.3f}", int(R['lsg_kept'][i]),
                        int(sel_mask[i]), int(R['inlier'][i]), "" if np.isnan(r) else f"{r:.3f}"])
    scalar = {k: (None if isinstance(v, float) and not np.isfinite(v) else v) for k, v in R['metrics'].items()}
    json.dump(dict(homography_src_to_ref_fullframe=None if R['H_full'] is None else R['H_full'].tolist(),
                   homography_cropped_frame=None if R['H'] is None else R['H'].tolist(),
                   src_crop_xyxy=list(R['src_crop']), ref_crop_xyxy=list(R['ref_crop']),
                   note="match_points.csv coordinates are in the ORIGINAL full images; registered.png is in the reference CROP frame",
                   metrics=scalar, hooks=hook_status(),
                   confidence=R['level'], notes=R['why'], config={k: v for k, v in cfg.items() if k != 'show'}),
              open(out / "report.json", "w"), indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    fig = build_dashboard(R)
    fig.savefig(out / "dashboard.png", dpi=110, bbox_inches='tight', facecolor='white')
    if cfg['show']:
        plt.show()
    plt.close(fig)
    zpath = shutil.make_archive(str(out), "zip", root_dir=out)
    return zpath

In [ ]:
# Cell 8b — the pipeline
def T(off):
    return np.array([[1, 0, off[0]], [0, 1, off[1]], [0, 0, 1.0]])

def run_pipeline(source_path, reference_path, cfg=None, src_crop=None, ref_crop=None, manual_pairs=None, meta=None, **overrides):
    """src_crop / ref_crop = (x0, y0, x1, y1) to cut away watermarks, text, frames (default: auto-trim of flat borders).
    manual_pairs = [((sx, sy), (rx, ry)), ...] — 4+ hand-picked correspondences in ORIGINAL full-image pixels (e.g. central
    peak, rim extremes). Optional rescue for pairs where illumination/viewpoint leave no reliable automatic features: it seeds
    the homography; automatic matches then refine it. Read coordinates off show_for_cropping().
    meta = optional dict of sensor metadata handed to the stage hooks (Cell 7b)."""
    cfg = {**CFG, **(cfg or {}), **overrides}
    t0 = time.perf_counter()
    src_full, ref_full = read_image(source_path), read_image(reference_path)
    def crop_of(im, c):
        return tuple(c) if c is not None else (auto_trim(im) if cfg['auto_trim'] else (0, 0, im.shape[1], im.shape[0]))
    sc, rc = crop_of(src_full, src_crop), crop_of(ref_full, ref_crop)
    src, ref = src_full[sc[1]:sc[3], sc[0]:sc[2]], ref_full[rc[1]:rc[3], rc[0]:rc[2]]
    meta = meta or {}
    # ---- HOOK DEM_RENDER_RELIGHT (placeholder): relief + sun removed before matching
    rel = call_hook('DEM_RENDER_RELIGHT', src, ref, meta)
    src_m, ref_m = rel if rel else (src, ref)
    log(f"Source {src_full.shape[:2][::-1]} -> crop {sc} {src.shape[:2][::-1]} · Reference {ref_full.shape[:2][::-1]} -> crop {rc} {ref.shape[:2][::-1]}")

    # ---- working resolution (matching only; everything is mapped back to cropped-original pixels)
    a = min(1.0, cfg['work_max_side'] / max(src.shape[:2])); b = min(1.0, cfg['work_max_side'] / max(ref.shape[:2]))
    log('[hook] DEM_RENDER_RELIGHT: ' + ('INSTALLED -> matching on relit images' if rel else 'placeholder (not installed) -> matching on raw images'))
    src_w, _ = resize_by(src_m, a); ref_w, M_ref = resize_by(ref_m, b)

    # ---- HOOK METADATA_ALIGNMENT (placeholder) + stage 1: scale sweep
    mres = call_hook('METADATA_ALIGNMENT', meta, src.shape, ref.shape) or {}
    log('[hook] METADATA_ALIGNMENT: ' + ('INSTALLED' if mres else 'placeholder (not installed) -> blind scale sweep'))
    mode = cfg['viewpoint_sim']
    log("[1] Multi-scale SIFT sweep")
    plain_confident = None
    if mres.get('scale'):
        s, scale_conf = float(mres['scale']), True
        use_vp = mode in ('auto', 'always')
    else:
        if mode == 'always':
            s, scale_conf, _ = scale_sweep(src_w, ref_w, cfg, vp=True)
        else:
            s, scale_conf, _ = scale_sweep(src_w, ref_w, cfg, vp=False)
            plain_confident = scale_conf
            if not scale_conf and mode == 'auto':
                log("    plain sweep not confident -> retrying with simulated viewpoints (oblique pair?)")
                s, scale_conf, _ = scale_sweep(src_w, ref_w, cfg, vp=True)
        use_vp = (mode == 'always') or (mode == 'auto' and not plain_confident)
    log(f"    -> scale {s}x ({'confident' if scale_conf else 'NOT confident, falling back to 1.0x'}) · viewpoint simulation: {'ON' if use_vp else 'off'}")
    src_ws, M_src = resize_by(src_m, a * s)            # ONE resize from the original, exact map back
    g0, g1 = to_gray(src_ws, cfg['clahe']), to_gray(ref_w, cfg['clahe'])

    # ---- stage 2/3: candidates (deep + SIFT pool), all mapped to CROPPED-ORIGINAL coordinates
    log("[2-3] Candidate matching")
    parts, matcher = [], []
    tilts_d = cfg['tilts_deep'] if use_vp else (1.0,)
    res, info = deep_match_views(make_views(g0, tilts_d, cfg['rot_base'], cfg['max_views']), g1, cfg)
    if res is None:
        log(f"    ! {info} -> using RootSIFT candidates only")
    else:
        pa, pb, ca = res
        if use_vp and cfg['vp_both_directions']:
            res2, _ = deep_match_views(make_views(g1, tilts_d, cfg['rot_base'], cfg['max_views']), g0, cfg, swap=True)
            if res2 is not None:
                pa, pb, ca = np.vstack([pa, res2[0]]), np.vstack([pb, res2[1]]), np.concatenate([ca, res2[2]])
        if len(pa):
            parts.append((to_original(M_src, pa), to_original(M_ref, pb), ca, 'superpoint'))
        matcher.append("SuperPoint+LightGlue")
        log(f"    deep: {info['n_views']} view(s) on {info['device']} · raw matches {info['n_raw']}")
    if res is None or cfg['use_sift_pool']:
        sm = SiftMatcher(cfg['sift_nfeatures'], cfg['sift_ratio'])
        if use_vp:
            p0, p1, c = sift_vp_candidates(sm, g0, g1, cfg, cfg['tilts_sift'])
        else:
            p0, p1, c = sm.match(sm.feats(g0), sm.feats(g1))
        if len(p0):
            parts.append((to_original(M_src, p0), to_original(M_ref, p1), c, 'sift'))
        matcher.append("RootSIFT" + ("+viewpoint-sim" if use_vp else ""))
    C0, C1, conf, lab = merge_candidates(parts, cfg['dedupe_px'])
    if len(C0) > cfg['max_candidates']:
        top = np.argsort(-conf)[:cfg['max_candidates']]
        C0, C1, conf, lab = C0[top], C1[top], conf[top], lab[top]
    n_cand = len(C0)
    log(f"    candidate pool: {n_cand} matches ({', '.join(matcher)})")

    R = dict(cfg=cfg, src=src, ref=ref, src_name=Path(source_path).name, ref_name=Path(reference_path).name,
             src_crop=sc, ref_crop=rc, C0=C0, C1=C1, conf=conf, lab=lab, n_candidates=n_cand, scale=s,
             scale_confident=scale_conf, matcher=" + ".join(matcher), vp_note="ON" if use_vp else "off",
             H=None, H_full=None, reg=None, reg_h=None, sel_idx=np.array([], int),
             lsg_score=np.zeros(n_cand), lsg_kept=np.zeros(n_cand, bool), lsg_ok=True, n_lsg=0,
             inlier=np.zeros(n_cand, bool), resid=np.full(n_cand, np.nan), n_inliers=0, n_inliers_all=0,
             inlier_ratio=0.0, rmse=float('nan'), cv_rmse=float('nan'), med_err=float('nan'), p95_err=float('nan'),
             max_err=float('nan'), err_pct=float('nan'), ssim=float('nan'), ncc=float('nan'),
             uniformity=dict(coverage=0.0, hull_ratio=0.0, clark_evans=float('nan'), score=0.0),
             refine_note="n/a", H_ok=False, H_msg="no homography estimated", tier_px=cfg['px_tiers'][0],
             nr=None, nr_used=False, nr_note="n/a", model_note="n/a", chance=None)
    if n_cand < 8:
        R['H_msg'] = f"only {n_cand} candidate matches"
        return _finish(R, t0, cfg)

    # ---- stage 4: LSG in a coarse-H-normalised frame, then uniform selection
    log("[4] Lunar Structural Graph filtering")
    Q0 = C0
    Hc, mc = cv2.findHomography(C0.astype(np.float32), C1.astype(np.float32), cv2.USAC_MAGSAC, cfg['coarse_px'],
                                maxIters=10000, confidence=0.999)
    if Hc is not None and int(mc.sum()) >= 8 and check_homography(Hc, src.shape)[0]:
        Q0 = apply_H(Hc, C0)
        log(f"    coarse H ({cfg['coarse_px']:.0f}px): {int(mc.sum())} loose inliers -> LSG runs in the H-normalised frame")
    else:
        log("    no usable coarse H -> LSG runs on raw coordinates (perspective may over-prune)")
    L = lsg_filter(Q0, C1, cfg)
    R['lsg_ok'], R['lsg_score'] = L['ok'], L['score']
    R['lsg_kept'][L['keep_idx']] = True
    R['n_lsg'] = len(L['keep_idx'])
    log(f"    {n_cand} -> {R['n_lsg']} matches in {L['iters']} rounds" + ("" if L['ok'] else "  ⚠ LSG FAILED: bypassed, using all matches"))
    comb = 0.5 * np.clip(conf, 0, 1) + 0.5 * R['lsg_score']
    kept = L['keep_idx']
    sel = kept[select_uniform(C1[kept], comb[kept], ref.shape, cfg['grid'], cfg['max_per_cell'])]
    R['sel_idx'] = sel
    log(f"    uniform selection -> {len(sel)} matches")
    if len(sel) < 8:
        R['H_msg'] = f"only {len(sel)} matches survived LSG / selection"
        return _finish(R, t0, cfg)

    # ---- stage 5: model selection — MAGSAC++ and spread-RANSAC (and manual seed), accuracy tiers, coverage-aware score
    P0, P1 = C0[sel], C1[sel]
    cover = lambda m: occupancy(P0[m], src.shape, cfg['cov_grid'])
    hyps = {}                                  # name -> (th, H, inlier_mask)
    seed = None
    if manual_pairs:
        mp = np.float64(manual_pairs).reshape(-1, 2, 2)
        ms = mp[:, 0] - np.float64(sc[:2]); mr = mp[:, 1] - np.float64(rc[:2])
        if len(mp) >= 4:
            seed, _ = cv2.findHomography(ms, mr, 0)
        elif len(mp) == 3:
            A_, _ = cv2.estimateAffine2D(ms.astype(np.float32), mr.astype(np.float32), method=cv2.LMEDS)
            seed = None if A_ is None else np.vstack([A_, [0, 0, 1]])
        if seed is not None and check_homography(seed, src.shape)[0]:
            ths_m = (max(cfg['coarse_px'], 16.0), 8.0, 4.0, 2.0)
            Hm = refine_from_seed(seed, np.vstack([C0[sel], np.repeat(ms, 5, 0)]), np.vstack([C1[sel], np.repeat(mr, 5, 0)]), ths_m)
            for th in cfg['px_tiers']:
                hyps[('manual', th)] = (th, Hm, np.linalg.norm(apply_H(Hm, P0) - P1, axis=1) <= th)
            log(f"[5] manual seed from {len(mp)} correspondences, refined with automatic matches")
        else:
            log("[5] manual_pairs given but unusable (need >=3, valid geometry) — ignored")
    for th in cfg['px_tiers']:
        Ht, mk = cv2.findHomography(P0.astype(np.float32), P1.astype(np.float32), cv2.USAC_MAGSAC, th,
                                    maxIters=10000, confidence=0.999)
        if Ht is not None and check_homography(Ht, src.shape)[0]:
            hyps[('magsac', th)] = (th, Ht, mk.ravel().astype(bool))
        sp = spread_ransac(P0, P1, src.shape, th, cfg['spread_iters'], cfg['cov_grid'])
        if sp is not None:
            hyps[('spread', th)] = (th, sp[0], sp[1])
    # ---- HOOK PROPOSE_ALIGNMENTS (placeholder): independent global alignments join the same scoring
    props = call_hook('PROPOSE_ALIGNMENTS', src, ref, meta, default=[]) or []
    log('[hook] PROPOSE_ALIGNMENTS: ' + (f'INSTALLED -> {len(props)} proposal(s)' if 'PROPOSE_ALIGNMENTS' in HOOKS else 'placeholder (not installed)'))
    for pname, Hx in props:
        if Hx is not None and check_homography(Hx, src.shape)[0]:
            for th in cfg['px_tiers']:
                hyps[(pname, th)] = (th, Hx, np.linalg.norm(apply_H(Hx, P0) - P1, axis=1) <= th)
    if not hyps:
        R['H_msg'] = "no valid homography hypothesis at any tier"
        return _finish(R, t0, cfg)
    rows = []
    for (name, th), (_th, Ht, m) in hyps.items():
        n = int(m.sum()); cv_ = cover(m) if n else 0.0
        rows.append((name, th, n, cv_, cov_score(n, cv_)))
        log(f"[5] {name:7s} @ {th:>4g}px: {n:>3}/{len(sel)} inliers · source-grid coverage {cv_:.0%} · score {cov_score(n, cv_):.1f}")
    def pick(rows_):
        ok_ = [r for r in rows_ if r[2] >= cfg['min_tier_inliers'] and r[3] >= cfg['min_cov']]
        if ok_:                                           # strictest tier that has enough inliers AND coverage; best score within it
            th0 = min(r[1] for r in ok_)
            return max((r for r in ok_ if r[1] == th0), key=lambda r: r[4])
        return max(rows_, key=lambda r: r[4])
    man = [r for r in rows if r[0] == 'manual']
    name, th, n, cv_, _sc = pick(man) if man else pick(rows)
    _th, H, inl = hyps[(name, th)]
    inl = np.asarray(inl, bool)
    R['tier_px'] = th
    R['model_note'] = f"{name} @ {th:g}px ({n} inliers, {cv_:.0%} coverage)"
    log(f"    -> chosen: {R['model_note']}")
    if man:
        pm = np.vstack([ms]); R['C0'] = np.vstack([R['C0'], ms]); R['C1'] = np.vstack([R['C1'], mr])
        R['conf'] = np.concatenate([R['conf'], np.ones(len(ms))]); R['lab'] = np.concatenate([R['lab'], np.array(['manual'] * len(ms), dtype=object)])
        R['lsg_score'] = np.concatenate([R['lsg_score'], np.ones(len(ms))]); R['lsg_kept'] = np.concatenate([R['lsg_kept'], np.ones(len(ms), bool)])
        R['inlier'] = np.concatenate([R['inlier'], np.zeros(len(ms), bool)]); R['resid'] = np.concatenate([R['resid'], np.full(len(ms), np.nan)])
        C0, C1 = R['C0'], R['C1']
        R['sel_idx'] = sel = np.concatenate([sel, np.arange(len(C0) - len(ms), len(C0))])
        R['n_candidates'] = len(C0)
        P0, P1 = C0[sel], C1[sel]
        inl = np.concatenate([inl, np.ones(len(ms), bool)])

    # ---- stage 6: Huber refinement (only when enough inliers to be meaningful)
    f = cfg['huber_fscale'] * max(1.0, th / 2)
    if inl.sum() >= cfg['min_refine_inliers']:
        try:
            Hr, _r = refine_huber(H, P0[inl], P1[inl], f)
            if check_homography(Hr, src.shape)[0] and huber_cost(Hr, P0[inl], P1[inl], f) <= huber_cost(H, P0[inl], P1[inl], f):
                H, R['refine_note'] = Hr, f"Huber applied on {inl.sum()} inliers"
            else:
                R['refine_note'] = "Huber rejected (no improvement / invalid) — MAGSAC++ H kept"
        except Exception as e:
            R['refine_note'] = f"Huber failed ({type(e).__name__}) — MAGSAC++ H kept"
    else:
        R['refine_note'] = f"skipped: {inl.sum()} inliers < {cfg['min_refine_inliers']} (would over-fit 8 DoF)"
    log(f"[6] {R['refine_note']}")
    R['H'] = H
    ok, msg, quad = check_homography(H, src.shape)
    R['H_ok'], R['H_msg'] = ok, msg
    # ---- HOOK CHANCE_VERIFICATION (placeholder): is the agreement statistically meaningful?
    chance = call_hook('CHANCE_VERIFICATION', P0, P1, H, src.shape, ref.shape)
    R['chance'] = chance
    log('[hook] CHANCE_VERIFICATION: ' + (f'INSTALLED -> {chance}' if chance else 'placeholder (not installed) -> thresholds are heuristic'))
    if chance and not chance.get('accept', True):
        ok, R['H_ok'], R['H_msg'] = False, False, 'rejected by chance verification (agreement not statistically meaningful)'

    # ---- stage 7: non-rigid TPS correction (relief / parallax a homography cannot model)
    if ok and cfg['nr_enable']:
        loose = np.linalg.norm(apply_H(H, P0) - P1, axis=1) <= 2 * th
        cov_nr = occupancy(P0[loose], src.shape, cfg['cov_grid'])
        nr = nonrigid_fit(H, P0[loose], P1[loose], ref.shape, cfg) if cov_nr >= cfg['min_cov'] else None
        if cov_nr < cfg['min_cov']:
            R['nr_note'] = f"skipped: control points cover only {cov_nr:.0%} of the source (TPS would extrapolate)"
        elif nr is None:
            R['nr_note'] = f"skipped (<{cfg['nr_min_pts']} control points)"
        else:
            R['nr'], R['nr_used'] = nr, nr['use']
            R['nr_note'] = (f"USED — held-out {nr['cv_h']:.2f}px -> {nr['cv_nr']:.2f}px ({nr['n']} pts)" if nr['use']
                            else f"rejected — held-out {nr['cv_h']:.2f}px (H) vs {nr['cv_nr']:.2f}px (TPS), no clear gain")
        log(f"[7] Non-rigid: {R['nr_note']}")
    # ---- HOOK DENSE_TIE_POINTS (placeholder): uniform lattice tie points after coarse alignment
    dt = call_hook('DENSE_TIE_POINTS', src, ref, H) if ok else None
    log('[hook] DENSE_TIE_POINTS: ' + ('INSTALLED' if 'DENSE_TIE_POINTS' in HOOKS else 'placeholder (not installed)'))
    if dt is not None and len(dt[0]):
        d0, d1, dc = (np.asarray(x, float) for x in dt); k0, nn = len(R['C0']), len(d0)
        R['C0'], R['C1'] = np.vstack([R['C0'], d0]), np.vstack([R['C1'], d1]); R['conf'] = np.concatenate([R['conf'], dc])
        R['lab'] = np.concatenate([R['lab'], np.array(['lattice'] * nn, dtype=object)])
        R['lsg_score'] = np.concatenate([R['lsg_score'], np.ones(nn)]); R['lsg_kept'] = np.concatenate([R['lsg_kept'], np.ones(nn, bool)])
        R['inlier'] = np.concatenate([R['inlier'], np.zeros(nn, bool)]); R['resid'] = np.concatenate([R['resid'], np.full(nn, np.nan)])
        R['sel_idx'] = np.concatenate([R['sel_idx'], np.arange(k0, k0 + nn)]); R['n_candidates'] = len(R['C0'])
    return _finish(R, t0, cfg, quad)

def _finish(R, t0, cfg, quad=None):
    """Metrics, warp, verdict, export."""
    ref, src, H = R['ref'], R['src'], R['H']
    th = R['tier_px']
    if H is not None and R['H_ok']:
        R['resid'] = np.linalg.norm(apply_H(H, R['C0']) - R['C1'], axis=1)
        R['inlier'] = R['resid'] <= th
        sel = R['sel_idx']
        ins = sel[R['inlier'][sel]]
        R['n_inliers'], R['n_inliers_all'] = len(ins), int(R['inlier'].sum())
        R['inlier_ratio'] = len(ins) / max(len(sel), 1)
        if len(ins):
            e = R['resid'][ins]
            R['rmse'], R['med_err'] = float(np.sqrt(np.mean(e ** 2))), float(np.median(e))
            R['p95_err'], R['max_err'] = float(np.percentile(e, 95)), float(e.max())
            R['err_pct'] = 100 * R['rmse'] / float(np.hypot(*ref.shape[:2][::-1]))
            R['cv_rmse'] = crossval_rmse(R['C0'][ins], R['C1'][ins], cfg['cv_folds'])
            fp = np.zeros(ref.shape[:2], np.uint8)
            cv2.fillConvexPoly(fp, np.round(quad).astype(np.int32), 1)
            R['uniformity'] = uniformity(R['C1'][ins], fp)
        R['reg_h'], _v, ssim_h, ncc_h = image_quality(src, ref, H, None)
        if R['nr_used']:
            R['reg'], _v, R['ssim'], R['ncc'] = image_quality(src, ref, H, R['nr'])
        else:
            R['reg'], R['ssim'], R['ncc'] = R['reg_h'], ssim_h, ncc_h
        R['valid'] = _v
        R['H_full'] = T(R['ref_crop'][:2]) @ H @ T((-R['src_crop'][0], -R['src_crop'][1]))
    is_man = np.asarray(R['lab'] == 'manual') if len(R['lab']) else np.zeros(0, bool)
    man_n = int(is_man.sum())
    man_rmse = float(np.sqrt(np.mean(R['resid'][is_man] ** 2))) if (man_n and H is not None and R['H_ok']) else float('nan')
    n_auto = int((R['inlier'] & ~is_man).sum()) if R['H_ok'] else 0
    R['manual_n'], R['manual_rmse'], R['n_auto_inliers'] = man_n, man_rmse, n_auto
    m = dict(manual_n=man_n, manual_rmse=man_rmse, n_auto_inliers=n_auto,
             H_ok=R['H_ok'], H_msg=R['H_msg'], n_inliers=R['n_inliers'], inlier_ratio=R['inlier_ratio'],
             cv_rmse=R['cv_rmse'], uniformity=R['uniformity'], lsg_ok=R['lsg_ok'], scale_confident=R['scale_confident'],
             tier_px=th, nr_used=R['nr_used'], nr_cv_h=(R['nr'] or {}).get('cv_h', float('nan')),
             nr_cv_nr=(R['nr'] or {}).get('cv_nr', float('nan')))
    R['level'], R['why'] = assess(m)
    R['time'] = time.perf_counter() - t0
    u = R['uniformity']; nrd = R['nr'] or {}
    R['metrics'] = dict(n_candidates=R['n_candidates'], n_after_lsg=R['n_lsg'], n_final_set=int(len(R['sel_idx'])),
                        accuracy_tier_px=th, n_inliers=R['n_inliers'], inlier_ratio=R['inlier_ratio'],
                        n_inliers_all_candidates=R['n_inliers_all'],
                        rmse_px=R['rmse'], heldout_cv_rmse_px=R['cv_rmse'], median_err_px=R['med_err'], p95_err_px=R['p95_err'],
                        max_err_px=R['max_err'], err_pct_of_diagonal=R['err_pct'], ssim=R['ssim'], ncc=R['ncc'],
                        uniformity_score=u['score'], grid_coverage=u['coverage'], hull_ratio=u['hull_ratio'],
                        clark_evans=u['clark_evans'], scale=R['scale'], scale_confident=R['scale_confident'],
                        lsg_ok=R['lsg_ok'], manual_points=R['manual_n'], manual_rmse_px=R['manual_rmse'], auto_inliers=R['n_auto_inliers'], model=R['model_note'], refinement=R['refine_note'], nonrigid=R['nr_note'], nonrigid_used=R['nr_used'],
                        nonrigid_cv_h_px=nrd.get('cv_h'), nonrigid_cv_tps_px=nrd.get('cv_nr'),
                        viewpoint_sim=R['vp_note'], matcher=R['matcher'], seconds=R['time'])
    log(f"\n=== CONFIDENCE: {R['level']} === " + "; ".join(R['why']))
    R['zip'] = export_results(R, cfg['out_dir'])
    log(f"Outputs written to '{cfg['out_dir']}/' and {R['zip']}")
    return R

In [ ]:
# ===== CELL 8c: RUN (uses source_path + reference_path + meta from Cell I4 - no upload here) =====
print("Source   :", source_path)
print("Reference:", reference_path)

R = run_pipeline(source_path, reference_path, meta=meta)
# If auto-trim misses a watermark/text/frame:  R = run_pipeline(source_path, reference_path, src_crop=(x0,y0,x1,y1), ref_crop=(x0,y0,x1,y1))
# Hand-picked rescue (4+ pairs, ORIGINAL pixels):  R = run_pipeline(source_path, reference_path, manual_pairs=[((sx,sy),(rx,ry)), ...])

try:
    from google.colab import files
    files.download(R['zip'])
except Exception as e:
    print("Auto-download skipped; open the folder panel and download:", R.get('zip'), "|", e)


In [ ]:
# Cell 9 — optional self-test on a synthetic crater pair with a KNOWN transform
def make_synthetic_pair(seed=0, size=900, light_delta=35):
    """Procedural crater terrain; reference lit from one azimuth, source (lower-res, rotated, mild perspective)
    lit from another + gamma + noise. Returns (src_bgr, ref_bgr, H_true src->ref)."""
    rng = np.random.default_rng(seed)
    h = np.zeros((size, size), np.float32)
    for _ in range(260):
        cx, cy = rng.uniform(0, size, 2); r = float(min(rng.lognormal(3.0, 0.7), 110))
        x0, x1, y0, y1 = int(max(cx - 1.5 * r, 0)), int(min(cx + 1.5 * r, size)), int(max(cy - 1.5 * r, 0)), int(min(cy + 1.5 * r, size))
        if x1 <= x0 or y1 <= y0: continue
        yy, xx = np.mgrid[y0:y1, x0:x1].astype(np.float32)
        d = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2) / r; depth = 0.25 * r
        h[y0:y1, x0:x1] += np.where(d < 1, -(1 - d ** 2) * depth, 0) + np.where((d >= 1) & (d < 1.4), 0.3 * depth * np.exp(-((d - 1) / 0.15) ** 2), 0)
    for sg, amp in [(2, 1.5), (6, 4), (18, 10)]:
        h += amp * cv2.GaussianBlur(rng.normal(size=h.shape).astype(np.float32), (0, 0), sg)
    gy, gx = np.gradient(h)
    def shade(az, elev=35):
        lx, ly = np.cos(np.radians(az)), np.sin(np.radians(az))
        v = 0.5 + 0.9 * np.sin(np.radians(elev)) * (-(gx * lx + gy * ly)) / (1 + 0.3 * np.abs(gx) + 0.3 * np.abs(gy))
        return np.clip(v, 0, 1)
    A, B = shade(200), shade(200 + light_delta)                      # two illumination directions
    ref_g = (255 * A[100:800, 80:780] ** 1.0).astype(np.uint8)
    src_full = (255 * (B ** 1.4) * np.linspace(0.75, 1.1, size)[None, :]).clip(0, 255).astype(np.uint8)[100:800, 80:780]
    c = np.array([350.0, 350.0]); th = np.radians(12); sc = 0.4
    Rm = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]]) * sc
    A3 = np.eye(3); A3[:2, :2] = Rm; A3[:2, 2] = np.array([140.0, 140.0]) - Rm @ c
    P = np.array([[1, 0, 0], [0, 1, 0], [1.5e-4, -1.0e-4, 1.0]])
    Hs = A3 @ P                                          # ref -> src
    src_g = cv2.warpPerspective(src_full, Hs, (280, 280), flags=cv2.INTER_AREA)
    src_g = np.clip(src_g.astype(np.float32) + rng.normal(0, 3, src_g.shape), 0, 255).astype(np.uint8)
    H_true = np.linalg.inv(Hs); H_true /= H_true[2, 2]
    return cv2.cvtColor(src_g, cv2.COLOR_GRAY2BGR), cv2.cvtColor(ref_g, cv2.COLOR_GRAY2BGR), H_true

def run_selftest(seed=0, light_delta=35, **overrides):
    """light_delta = difference in sun azimuth (deg) between the two images. ~35 is moderate; 100 is brutal."""
    src, ref, H_true = make_synthetic_pair(seed, light_delta=light_delta)
    os.makedirs("selftest_in", exist_ok=True)
    cv2.imwrite("selftest_in/source.png", src); cv2.imwrite("selftest_in/reference.png", ref)
    R = run_pipeline("selftest_in/source.png", "selftest_in/reference.png", out_dir="selftest_out", **overrides)
    if R['H'] is not None:
        gx, gy = np.meshgrid(np.linspace(10, src.shape[1] - 10, 12), np.linspace(10, src.shape[0] - 10, 12))
        pts = np.c_[gx.ravel(), gy.ravel()]
        err = np.linalg.norm(apply_H(R['H'], pts) - apply_H(H_true, pts), axis=1)
        print(f"\nGROUND-TRUTH CHECK: mean {err.mean():.2f} px · max {err.max():.2f} px (in reference pixels, over a 12x12 grid)")
        R['gt_err'] = err
    else:
        print("\nGROUND-TRUTH CHECK: no homography produced")
    return R

# R_test = run_selftest()     # <- uncomment to verify the whole pipeline without your own images